# LiteLLM End to End: Standalone, with Strands, with LangChain and LangGraph

**Running case: Torchlight Roadside Assistance, the Dispatch Copilot.**

You already know Strands, LangChain and LangGraph basics. This notebook is about the layer *underneath* them: the thing that decides **which model actually gets the call, what happens when it fails, and who pays for it.**

Every section follows the same beat:

| Beat | What happens |
|---|---|
| **Scenario** | A concrete Torchlight situation |
| **Challenge** | 3 options, you pick before scrolling |
| **Verdict** | Which option and why the others break |
| **LLD** | Diagram of the piece you are about to build |
| **Code** | Runnable, verified against `litellm 1.95.0` |
| **Probe** | A cell that proves the behaviour, not just asserts it |

Nothing here is pseudo code. Every API in this notebook was executed against the installed packages before it was written.

## 0.1 How to run this

**Everything runs offline by default.** There is a `MOCK = True` switch in the setup cell. With `MOCK = True` no AWS credentials are needed and no money is spent, because LiteLLM's `mock_response` intercepts the call before the provider is contacted. Flip it to `False` to hit real Bedrock.

**VS Code**

```bash
python -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install "litellm==1.95.0" "strands-agents[litellm]" langchain langgraph langchain-litellm
aws configure                       # only needed when MOCK = False
```
Then open this file and select `.venv` as the kernel.

**Google Colab**

```python
!pip install -q "litellm==1.95.0" "strands-agents[litellm]" langchain langgraph langchain-litellm
import os
os.environ["AWS_REGION_NAME"] = "us-east-1"   # only needed when MOCK = False
```

> **Bedrock gotcha carried over from earlier sessions:** LiteLLM reads `AWS_REGION_NAME`, not `AWS_REGION`. And LiteLLM model strings need the `bedrock/` prefix *plus* the `us.` inference profile prefix: `bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0`. Strands `BedrockModel` takes the bare `us.anthropic...` with no `bedrock/`. Two different rules, same model.

## 0.2 The problem, before the tool

Torchlight runs a national roadside desk. One month of real traffic looks like this:

| Class | Share | Example request | Latency budget | What a wrong answer costs |
|---|---|---|---|---|
| **A. Status lookup** | 55% | "where is my technician?" | under 1s | annoyance |
| **B. Fault triage** | 25% | "grinding noise, smoke from the bonnet" | under 3s | a wasted van dispatch |
| **C. Coverage adjudication** | 12% | read the 60 page fleet contract, decide who pays | under 20s | real money, disputes |
| **D. Safety critical** | 5% | "motorway hard shoulder, two kids in the car, dark" | under 3s | someone gets hurt |
| **E. Photo diagnosis** | 3% | photo of a dashboard warning cluster | under 5s | wrong parts on the van |

Now do the arithmetic that kills most pilots. 1.2 million requests a month, average 800 input + 300 output tokens.

| Approach | Monthly model spend | Why |
|---|---|---|
| Everything on Claude Sonnet 4.5 | around $8,300 | 1.2M x (800 x 3.3e-6 + 300 x 1.65e-5) |
| Everything on Nova Micro | around $84 | 1.2M x (800 x 3.5e-8 + 300 x 1.4e-7) |
| Everything on Nova Micro | **but class C and D are wrong often enough to matter** | a 5% class D error rate is not a cost line, it is an incident |

That table is the whole argument. **The cheap model is 99x cheaper and unacceptable for 17% of traffic. The strong model is correct everywhere and 99x too expensive for 80% of traffic.** Neither single choice survives contact with the traffic mix.

So the real system needs four things at once:

1. **One call shape** so the application code does not fork per provider.
2. **A decision** about which model serves which request.
3. **A plan for when that model is down, throttled, over its context, or refuses.**
4. **A place to see cost, latency and failure per class**, or you are flying blind.

LiteLLM is the layer that gives you all four. Not an agent framework. Not a prompt library. The layer between your agent and the metal.

## 0.3 Mental model: LiteLLM wears three hats

This is the single most useful thing to hold in your head. LiteLLM is three products that share a name, and people confuse them constantly.

| Hat | What it is | Lives where | Solves |
|---|---|---|---|
| **The Translator** (SDK) | `litellm.completion()` | inside your process, one function | 100+ providers, one request and response shape |
| **The Dispatcher** (Router) | `litellm.Router` | inside your process, a Python object | which deployment, load balancing, retries, fallbacks, cooldowns |
| **The Terminal** (Proxy / Gateway) | `litellm --config config.yaml` | a separate HTTP service | keys, budgets, teams, guardrails, central logging, one URL for every app |

An airport version of the same idea, because Torchlight has a fleet and it maps cleanly:

```mermaid
flowchart LR
    A[Your app] --> T[Translator<br/>one boarding pass format<br/>for every airline]
    T --> D[Dispatcher<br/>air traffic control<br/>which runway, divert on failure]
    D --> G[Terminal<br/>one entrance, security,<br/>billing, gate logs]
    G --> M1[Nova Micro]
    G --> M2[Nova Lite]
    G --> M3[Claude Haiku 4.5]
    G --> M4[Claude Sonnet 4.5]
```

**The rule that resolves most confusion:**

- Translator without Dispatcher is fine for a script.
- Dispatcher without Terminal is fine for **one** service owned by **one** team.
- The moment a second team, a second app, or a finance person asks "who spent what", you need the Terminal.

You can use all three, or any prefix of the three. You cannot skip the Translator, because the other two are built on it.

## 0.4 HLD: the whole Torchlight Dispatch Copilot

This is the target. Every later section builds exactly one box, and the section number is on the box.

```mermaid
flowchart TB
    subgraph CH[Channels]
        APP[Consumer app]
        FLEET[B2B fleet portal]
        OPS[Internal ops console]
    end

    subgraph GW[LiteLLM Proxy / Gateway - S9]
        KEYS[Virtual keys<br/>per team budget<br/>rate limits]
        GUARD[Guardrails<br/>PII, refusal policy]
    end

    subgraph BRAIN[Decision layer - S7]
        CLS[Classifier cascade<br/>heuristic to intent to LLM]
        POL[Routing policy<br/>class to model group]
    end

    subgraph RT[LiteLLM Router - S6]
        LB[Load balancing<br/>weights, least busy, latency]
        FB[Fallback ladder<br/>group, context, content policy]
        CD[Cooldowns and retries]
    end

    subgraph FLEETM[Model groups]
        FAST[tl-fast<br/>Nova Micro x2 regions]
        BAL[tl-balanced<br/>Nova Lite 300k, vision]
        DEEP[tl-deep<br/>Haiku 4.5, tools]
        GUARDM[tl-guard<br/>Sonnet 4.5, safety only]
    end

    subgraph AG[Agent runtimes]
        STR[Strands agent - S10]
        LG[LangGraph desk - S11]
    end

    subgraph OBS[Plugins - S8]
        LEDGER[Cost ledger and budget brake]
        REDACT[PII redaction]
        TRACE[Failure and fallback trace]
    end

    APP --> KEYS
    FLEET --> KEYS
    OPS --> KEYS
    KEYS --> GUARD
    GUARD --> CLS
    CLS --> POL
    POL --> STR
    POL --> LG
    STR --> LB
    LG --> LB
    LB --> FB
    FB --> CD
    CD --> FAST
    CD --> BAL
    CD --> DEEP
    CD --> GUARDM
    RT -.emits.-> OBS
```

**Section map.** S4 the SDK. S5 failure taxonomy. S6 Router. S7 the decision layer. S8 plugins. S9 proxy and gateway. S10 Strands. S11 LangChain and LangGraph. S12 the whole thing wired and failure tested.

In [ ]:
# ============================================================
# SETUP. Run this once. Everything below depends on it.
# ============================================================
import os, json, time, math, random, asyncio, logging, textwrap
from dataclasses import dataclass, field
from typing import Any, Optional

import litellm
from litellm import Router

# MOCK = True  -> no AWS needed, no spend, deterministic outputs.
# MOCK = False -> real Bedrock calls. Set AWS creds and AWS_REGION_NAME first.
MOCK = True

os.environ.setdefault("AWS_REGION_NAME", "us-east-1")   # LiteLLM reads THIS, not AWS_REGION

litellm.suppress_debug_info = True
litellm.drop_params = True          # silently drop params a provider does not support
for _n in ["LiteLLM", "LiteLLM Router", "LiteLLM Proxy", "httpx"]:
    logging.getLogger(_n).setLevel(logging.CRITICAL)

# The Torchlight fleet. Real Bedrock IDs, real LiteLLM prefixes.
MICRO  = "bedrock/us.amazon.nova-micro-v1:0"
LITE   = "bedrock/us.amazon.nova-lite-v1:0"
HAIKU  = "bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET = "bedrock/us.anthropic.claude-sonnet-4-5-20250929-v1:0"

def mock(resp):
    """Return kwargs that force a canned response when MOCK is on, else nothing."""
    return {"mock_response": resp} if MOCK else {}

def rule(title=""):
    print("\n" + "=" * 62)
    if title:
        print(title)
        print("=" * 62)

print(f"litellm ready. MOCK={MOCK}")
print("fleet:", *[m.split('/')[-1] for m in (MICRO, LITE, HAIKU, SONNET)], sep="\n  ")

---
# S4. The Translator: LiteLLM as a plain SDK

**What this section buys you:** one function signature that works across every provider Torchlight might ever use, plus the metadata you need to make routing decisions later.

Skip nothing here. Sections 6 and 7 are just this call with a decision in front of it.

### 4.1 One call shape, any model

Anatomy of a LiteLLM model string:

```
bedrock / us. anthropic.claude-haiku-4-5-20251001-v1:0
  |        |         |
  |        |         +-- the provider's own model id
  |        +------------ cross region inference profile prefix (Bedrock requirement)
  +--------------------- LiteLLM provider route
```

Change the string, keep the code. That is the entire promise of the Translator hat.

In [ ]:
resp = litellm.completion(
    model=MICRO,
    messages=[
        {"role": "system", "content": "You are the Torchlight dispatch desk. Be terse."},
        {"role": "user",   "content": "Where is the technician for case TL-88421?"},
    ],
    max_tokens=120,
    temperature=0.2,
    **mock("Technician EN-14 is 22 minutes away, currently on the A34 northbound."),
)

print("text     :", resp.choices[0].message.content)
print("finish   :", resp.choices[0].finish_reason)
print("model     :", resp.model)
print("usage    :", resp.usage.prompt_tokens, "in /", resp.usage.completion_tokens, "out")

### 4.2 The response contract, and the part nobody reads

LiteLLM normalises every provider into the OpenAI response shape. That is well known. What matters operationally is `_hidden_params`, which is where the routing and cost evidence lives.

In [ ]:
rule("_hidden_params: the operational metadata")
for k in ["custom_llm_provider", "model_id", "response_cost", "_response_ms", "api_base"]:
    print(f"  {k:22s} {resp._hidden_params.get(k)}")

print("\nWhy each one matters:")
print("  custom_llm_provider  which provider actually served it (bedrock, anthropic, openai...)")
print("  model_id             WHICH DEPLOYMENT served it. Essential once the Router exists (S6).")
print("  response_cost        LiteLLM's own cost computation for this single call.")
print("  _response_ms         wall clock latency, the input to latency based routing.")

### 4.3 Cost and context: the numbers your routing policy runs on

LiteLLM ships a cost and capability map for thousands of models. You can query it **without making a call**, which means your routing decision costs nothing.

> **Verified gotcha.** `litellm.get_max_tokens(model)` returns the maximum **output** tokens, not the context window. For context window checks you want `get_model_info(model)["max_input_tokens"]`. For Haiku 4.5 those are 64,000 and 200,000. Using the wrong one silently breaks every long document routing rule you write.

In [ ]:
rule("Fleet capability and price sheet (no API call made)")
hdr = f"{'group':10s} {'in $/1M':>9s} {'out $/1M':>9s} {'ctx in':>9s} {'max out':>8s} {'tools':>6s} {'vision':>7s}"
print(hdr); print("-" * len(hdr))

FLEET = [("fast", MICRO), ("balanced", LITE), ("deep", HAIKU), ("guard", SONNET)]
INFO = {}
for label, m in FLEET:
    i = litellm.get_model_info(m)
    INFO[label] = i
    print(f"{label:10s} {i['input_cost_per_token']*1e6:9.2f} {i['output_cost_per_token']*1e6:9.2f} "
          f"{i['max_input_tokens']:>9,} {i['max_output_tokens']:>8,} "
          f"{str(bool(i.get('supports_function_calling'))):>6s} {str(bool(i.get('supports_vision'))):>7s}")

print("\nTrap check:")
print("  get_max_tokens(HAIKU)              =", litellm.get_max_tokens(HAIKU), " <- MAX OUTPUT")
print("  get_model_info(HAIKU)[max_input]   =", INFO['deep']['max_input_tokens'], "<- CONTEXT WINDOW")

In [ ]:
# Cost of one call, and cost of one month of Torchlight traffic.
rule("Cost math that decides the architecture")

def month_cost(model, n=1_200_000, tin=800, tout=300):
    i = litellm.get_model_info(model)
    return n * (tin * i["input_cost_per_token"] + tout * i["output_cost_per_token"])

for label, m in FLEET:
    print(f"  all 1.2M requests on {label:9s} -> ${month_cost(m):>10,.0f} / month")

print("\n  measured cost of the single call above: $", round(litellm.completion_cost(completion_response=resp), 8))
print("\n  Ratio guard-to-fast: %.0fx" % (month_cost(SONNET) / month_cost(MICRO)))
print("  That ratio is why every later section exists.")

### 4.4 Capability probes: the seed of capability routing

Before you write a single routing rule, LiteLLM can already answer "can this model even do the thing?". A router that ignores this produces a class of bug that is very hard to read in logs: the model does not error, it just quietly ignores your image or your tool schema.

In [ ]:
rule("Capability probes")
checks = [("tools", litellm.supports_function_calling), ("vision", litellm.supports_vision)]
print(f"{'model':10s} " + " ".join(f"{n:>8s}" for n, _ in checks))
for label, m in FLEET:
    print(f"{label:10s} " + " ".join(f"{str(f(m)):>8s}" for _, f in checks))

print("\nRead this table as routing constraints, not trivia:")
print("  class E (photo diagnosis) can never go to a model where vision is False.")
print("  a tool calling agent can never go to a model where tools is False.")
print("  these are HARD constraints. Cost and latency are SOFT preferences. Never mix them up.")

### 4.5 The offline switch: `mock_response`

This is the feature that makes LiteLLM teachable and testable, and almost nobody uses it.

| Form | Effect |
|---|---|
| `mock_response="some text"` | returns that text as a valid `ModelResponse`, with usage and cost |
| `mock_response="litellm.RateLimitError"` | **raises** a real `RateLimitError` |
| `mock_response="litellm.ContextWindowExceededError"` | raises that |
| `mock_response="litellm.InternalServerError"` | raises that |
| `mock_response=<an exception instance>` | raises exactly that instance, any type |
| `mock_tool_calls=[...]` | returns a tool call without a model |

That means your entire failure handling path is unit testable in CI with zero spend. Torchlight's on-call runbook can be **executed**, not just written.

> **Verified trap, and it will bite you.** Only the three string forms above raise. A string like `"litellm.Timeout"` or `"litellm.AuthenticationError"` does **not** raise, it comes back as literal text. For those, pass an exception *instance*.

In [ ]:
rule("Failure injection: what actually raises")

def probe(mr, label):
    try:
        r = litellm.completion(model=MICRO, messages=[{"role": "user", "content": "x"}], mock_response=mr)
        return f"NO RAISE, returned text: {r.choices[0].message.content[:40]!r}"
    except Exception as e:
        return f"raised {type(e).__name__}"

print("  string 'litellm.RateLimitError'      ->", probe("litellm.RateLimitError", ""))
print("  string 'litellm.Timeout'             ->", probe("litellm.Timeout", ""))
print("  instance litellm.Timeout(...)        ->",
      probe(litellm.Timeout(message="simulated", model=MICRO, llm_provider="bedrock"), ""))
print("  instance ContentPolicyViolationError ->",
      probe(litellm.ContentPolicyViolationError(message="simulated", model=MICRO, llm_provider="bedrock"), ""))

---
### Challenge 4.A

Torchlight's mobile app calls the copilot. Product wants a "typing" indicator that appears within 400ms even when the model takes 6 seconds. Your team proposes three designs.

| Option | Design |
|---|---|
| **A** | Keep `litellm.completion()`, show a spinner, accept the 6s wait |
| **B** | Switch to `litellm.completion(..., stream=True)` and forward chunks |
| **C** | Fire two calls in parallel, a fast model for a holding sentence and the real model for the answer, show whichever lands first |

Pick one. Consider: cost, correctness of what the user sees, and what happens on failure.

Scroll after you have committed.

### Verdict 4.A

**B**, with a caveat that rules out C.

- **A** is not wrong, it is just not an answer to the question. A spinner is not a first token.
- **C** doubles your call volume on 100% of traffic to improve perceived latency, and it introduces a genuinely bad failure mode: the holding sentence from the cheap model can **contradict** the real answer. On class D traffic ("is it safe to stay in the car?") a contradiction is a safety incident, not a UX blemish.
- **B** is the answer, and streaming is a first class citizen in LiteLLM: the same normalised chunk shape across providers.

**The caveat that matters more than the answer.** Streaming changes your error surface. A `RateLimitError` on a non streaming call arrives *before* you show anything. On a streaming call it can arrive **after** you have already rendered half a sentence. Fallbacks (S6) cannot un send those tokens. Torchlight's rule: stream to the consumer app, do not stream class C adjudications, because a half written coverage decision is worse than a slow one.

In [ ]:
rule("Streaming, and where the error surface moves")

stream = litellm.completion(
    model=MICRO,
    messages=[{"role": "user", "content": "One sentence: technician ETA for TL-88421."}],
    stream=True,
    **mock("Technician EN-14 arrives in about 22 minutes."),
)

first_token_at, t0, out = None, time.time(), []
for chunk in stream:
    piece = chunk.choices[0].delta.content or ""
    if piece and first_token_at is None:
        first_token_at = time.time() - t0
    out.append(piece)

print("first token after : %.3fs" % (first_token_at or 0))
print("assembled         :", "".join(out))
print("\nWhat changed: you committed to a response before you knew it would succeed.")
print("Rule for Torchlight: stream class A and B. Never stream class C or D.")

### Section 4 recap

| You learned | The artefact |
|---|---|
| One call shape across providers | `litellm.completion(model=..., messages=...)` |
| Where the operational truth lives | `resp._hidden_params` |
| Price and capability without a call | `get_model_info`, `supports_vision`, `supports_function_calling` |
| Context window is `max_input_tokens`, not `get_max_tokens` | the trap that breaks long document routing |
| Failure injection with zero spend | `mock_response`, instance form for full coverage |
| Streaming moves the error surface after the first token | the class C / class D rule |

**Everything from here is this call, with a decision in front of it and a safety net behind it.**

---
# S5. Failure is the design input, not the edge case

Most LLM systems are designed for the happy path and then have error handling bolted on. That ordering is backwards, and LiteLLM makes the correct ordering possible because it **normalises every provider's failures into one taxonomy**.

Without that, Torchlight's on-call engineer reads a Bedrock `ThrottlingException`, an OpenAI `429`, an Anthropic `overloaded_error` and a Vertex `ResourceExhausted` and has to know four vocabularies. With it, all four are `litellm.RateLimitError`.

### 5.1 The taxonomy, and the four R's

Every failure maps to exactly one of four responses. This table is the most portable thing in the notebook.

| LiteLLM exception | What actually happened | Response | Retrying is... |
|---|---|---|---|
| `RateLimitError` | throttled, capacity | **Retry** with backoff, then **Re-route** | correct |
| `Timeout` | the call took too long | **Retry** once, then **Re-route** | correct |
| `InternalServerError` | provider side 5xx | **Re-route** to another deployment | often useless on the same deployment |
| `ServiceUnavailableError` | provider down | **Re-route** | useless on the same deployment |
| `APIConnectionError` | network, DNS, TLS | **Retry** | correct |
| `ContextWindowExceededError` | your prompt is too big | **Reshape**: bigger model, or trim, or chunk | pointless, it will fail identically |
| `ContentPolicyViolationError` | the model refused | **Reshape** or **Refuse**, route to a differently aligned model | pointless |
| `AuthenticationError` | bad creds, wrong region, no model access | **Refuse**, page a human | actively harmful, it burns your retry budget |
| `BadRequestError` | your request is malformed | **Refuse**, it is a bug in your code | actively harmful |
| `NotFoundError` | wrong model id, no inference profile | **Refuse**, it is a config bug | actively harmful |

**Read the right hand column again.** Half of these must never be retried. A retry loop that treats all exceptions the same turns a config typo into 3x the latency and 3x the log noise, and still fails.

### 5.2 The inheritance trap, and it is worse than you think

Verified against `litellm 1.95.0`. This section will change code you have already written.

```mermaid
flowchart TB
    OE["openai.OpenAIError"] --> OAE["openai.APIError"]
    OAE --> OAS["openai.APIStatusError"]
    OAE --> OAC["openai.APIConnectionError"]
    OAE --> LAE["litellm.APIError"]
    OAS --> BR["litellm.BadRequestError"]
    OAS --> RL["litellm.RateLimitError"]
    OAS --> AU["litellm.AuthenticationError"]
    OAS --> IS["litellm.InternalServerError"]
    OAC --> TO["litellm.Timeout"]
    OAC --> LAC["litellm.APIConnectionError"]
    BR --> CW["litellm.ContextWindowExceededError"]
    BR --> CP["litellm.ContentPolicyViolationError"]
```

Read the arrows carefully. **The litellm classes are siblings under the openai classes, not children of each other.** Three consequences:

1. `ContextWindowExceededError` **is a** `litellm.BadRequestError`. Put `except BadRequestError` above the specific branch and your "route to a bigger model" logic is dead code.
2. `litellm.Timeout` is **not** a `litellm.APIConnectionError`. They are siblings. `except litellm.APIConnectionError` misses every timeout.
3. **`except litellm.APIError` catches nothing.** Not throttling, not timeouts, not 500s, not context overflow. It is a sibling of all of them. If you wrote it as a catch all, it has never caught anything.

The catch matrix below is generated, not typed. Read the `litellm.APIError` column.

In [ ]:
rule("The catch matrix. Generated, not asserted.")
import openai

mk = lambda C: C(message="simulated", model=HAIKU, llm_provider="bedrock")
EXCS = {
    "RateLimitError":              mk(litellm.RateLimitError),
    "Timeout":                     mk(litellm.Timeout),
    "InternalServerError":         mk(litellm.InternalServerError),
    "ContextWindowExceededError":  mk(litellm.ContextWindowExceededError),
    "ContentPolicyViolationError": mk(litellm.ContentPolicyViolationError),
    "AuthenticationError":         mk(litellm.AuthenticationError),
    "APIConnectionError":          mk(litellm.APIConnectionError),
}
CATCHERS = [
    ("litellm.APIError",           litellm.APIError),
    ("litellm.BadRequestError",    litellm.BadRequestError),
    ("litellm.APIConnectionError", litellm.APIConnectionError),
    ("openai.APIError",            openai.APIError),
]

print(f"  {'raised exception':30s}" + "".join(f"{n:>28s}" for n, _ in CATCHERS))
print("  " + "-" * (30 + 28 * len(CATCHERS)))
for name, e in EXCS.items():
    print(f"  {name:30s}" + "".join(f"{('CAUGHT' if isinstance(e, c) else '.'):>28s}" for _, c in CATCHERS))

print("\n  Column 1: `except litellm.APIError` catches NOTHING. Zero of seven.")
print("  Column 3: `except litellm.APIConnectionError` misses Timeout.")
print("  Column 4: `except openai.APIError` is the only true catch-all in this namespace.")
print("\n  Rules:")
print("    1. order except blocks MOST SPECIFIC FIRST")
print("    2. never use litellm.APIError as a catch-all")
print("    3. if you want a catch-all, use openai.APIError or bare Exception, deliberately")

In [ ]:
rule("The ordering trap, demonstrated")
from litellm.exceptions import BadRequestError, ContextWindowExceededError

boom = litellm.ContextWindowExceededError(message="prompt too long", model=HAIKU, llm_provider="bedrock")

print("  --- WRONG order (general first) ---")
try:
    raise boom
except BadRequestError:
    print("    caught as BadRequestError -> we will 'fix the malformed request'.")
    print("    WRONG. The request was well formed, it was just too big.")
except ContextWindowExceededError:
    print("    never reached")

print("\n  --- RIGHT order (specific first) ---")
try:
    raise boom
except ContextWindowExceededError:
    print("    caught correctly -> reshape: route to a 300k context deployment.")
except BadRequestError:
    print("    not reached, which is the point")

---
### Challenge 5.A

Torchlight's class C adjudication call starts failing at 02:00. The log line is:

```
litellm.RateLimitError: bedrock: Too many requests, please wait before trying again.
```

You have three proposals in the incident channel.

| Option | Proposal |
|---|---|
| **A** | Wrap the call in `for i in range(5): try/except: sleep(2**i)` |
| **B** | Set `num_retries=3` on the call and move on |
| **C** | Retry twice with jitter on the *same* deployment, then re-route to a different region or model, and only then fail |

Pick one before scrolling. The interesting part is not which is best, it is what each one is silently assuming.

### Verdict 5.A

**C**, and here is what A and B are assuming without saying so.

- **A** assumes the capacity will come back on the deployment you are hammering. At 02:00 with a regional Bedrock capacity event, it will not. You have built a 62 second latency spike that ends in the same error. And `2**i` with no jitter means every one of your pods retries **at the same instant**, which is a self inflicted thundering herd. If you take one thing from this section: **backoff without jitter is not backoff.**
- **B** is better and is the right first move, but `num_retries` in the SDK retries the **same model string**. It does not know about your other region. It fixes transient blips, not capacity events.
- **C** is the shape of the real answer, and it has a name in the reliability literature: **retry then hedge then shed**. Retry the transient case, hedge across independent capacity, shed load with a graceful answer rather than a stack trace.

**The punchline for this notebook:** you should not hand write C. C *is* the LiteLLM Router (S6). Writing it by hand once, as we do below, is how you understand what the Router is doing for you and when it is not enough.

### 5.3 LLD: the resilient call

```mermaid
flowchart TB
    START([call]) --> ATT[attempt on primary]
    ATT --> OK{success?}
    OK -- yes --> DONE([return])
    OK -- no --> CLS{classify exception}

    CLS -- RateLimit / Timeout / APIConnection --> BUDGET{retries left?}
    BUDGET -- yes --> JIT[sleep: base * 2^n * random jitter]
    JIT --> ATT
    BUDGET -- no --> REROUTE

    CLS -- InternalServer / ServiceUnavailable --> REROUTE[re-route: next deployment]
    CLS -- ContextWindowExceeded --> RESHAPE[reshape: larger context group]
    CLS -- ContentPolicy --> RESHAPE2[reshape: different alignment, or refuse]
    CLS -- Auth / BadRequest / NotFound --> REFUSE[refuse now, page a human]

    REROUTE --> LADDER{ladder exhausted?}
    RESHAPE --> LADDER
    RESHAPE2 --> LADDER
    LADDER -- no --> ATT
    LADDER -- yes --> DEGRADE[degraded answer<br/>+ human handoff]
    REFUSE --> DEGRADE
    DEGRADE --> DONE
```

Note the **grey terminal**: `DEGRADE`, not `raise`. For a roadside desk, "I could not answer, connecting you to a human, your case reference is TL-88421" is a successful outcome. An unhandled exception is not.

In [ ]:
# ============================================================
# S5. Hand built resilient call. This is what the Router does for you.
# Build it once by hand so the Router stops being magic.
# ============================================================

# ACTION is the four R's, as data instead of if statements.
# Note litellm.Timeout is listed EXPLICITLY. It is not a subclass of
# litellm.APIConnectionError, so leaving it out silently drops every timeout.
RETRYABLE  = (litellm.RateLimitError, litellm.Timeout, litellm.APIConnectionError)
REROUTABLE = (litellm.InternalServerError, litellm.ServiceUnavailableError)
FATAL      = (litellm.AuthenticationError, litellm.NotFoundError, litellm.PermissionDeniedError)

def classify(exc) -> str:
    """Map any LiteLLM exception to one of the four R's. ORDER MATTERS (see 5.2)."""
    if isinstance(exc, litellm.ContextWindowExceededError):   # before BadRequestError
        return "reshape_context"
    if isinstance(exc, litellm.ContentPolicyViolationError):  # before BadRequestError
        return "reshape_policy"
    if isinstance(exc, FATAL):
        return "refuse"
    if isinstance(exc, litellm.Timeout):                      # before APIConnectionError
        return "retry"
    if isinstance(exc, RETRYABLE):
        return "retry"
    if isinstance(exc, REROUTABLE):
        return "reroute"
    if isinstance(exc, litellm.BadRequestError):
        return "refuse"
    return "reroute"

@dataclass
class Attempt:
    model: str
    outcome: str
    detail: str = ""

def resilient_call(ladder, messages, max_retries=2, base=0.05, **kw):
    """
    ladder: [(model, purpose)] tried in order.
    Returns (response_or_None, trace).
    """
    trace: list[Attempt] = []
    for model, purpose in ladder:
        for n in range(max_retries + 1):
            try:
                r = litellm.completion(model=model, messages=messages, **kw)
                trace.append(Attempt(purpose, "success"))
                return r, trace
            except Exception as e:
                action = classify(e)
                trace.append(Attempt(purpose, action, type(e).__name__))
                if action == "retry" and n < max_retries:
                    # full jitter backoff. The random() is not decoration.
                    time.sleep(base * (2 ** n) * random.random())
                    continue
                break                      # any non retry action leaves the inner loop
        if trace[-1].outcome == "refuse":
            break                          # fatal, do not walk the rest of the ladder
    return None, trace

def show(trace, label):
    print(f"\n{label}")
    for a in trace:
        d = f"  ({a.detail})" if a.detail else ""
        print(f"    {a.model:16s} -> {a.outcome}{d}")

In [ ]:
rule("Running the resilient call against four injected failures")

LADDER = [(MICRO, "tl-fast"), (HAIKU, "tl-deep"), (SONNET, "tl-guard")]
MSG = [{"role": "user", "content": "Is the tow covered under fleet contract FL-2291?"}]

# 1. transient throttle on the first rung, then success
_, t = resilient_call(LADDER, MSG, mock_response="litellm.RateLimitError")
show(t, "1) RateLimitError everywhere: retries, re-routes down the ladder, ends exhausted")

# 2. context window exceeded: retrying is pointless, reshape immediately
_, t = resilient_call(LADDER, MSG, mock_response="litellm.ContextWindowExceededError")
show(t, "2) ContextWindowExceeded: NO retries burned, straight to reshape on each rung")

# 3. auth failure: stop instantly, do not walk the ladder
_, t = resilient_call(LADDER, MSG,
        mock_response=litellm.AuthenticationError(message="no bedrock access", model=MICRO, llm_provider="bedrock"))
show(t, "3) AuthenticationError: ONE attempt, then stop. Walking the ladder would fail identically.")

# 4. happy path
r, t = resilient_call(LADDER, MSG, mock_response="Covered under clause 7.2, excess GBP 50.")
show(t, "4) success on the first rung")
print("    answer:", r.choices[0].message.content)

### 5.4 The degraded answer

Notice that `resilient_call` returns `None` when everything fails. That is deliberate: **the transport layer should not invent an answer.** The application layer decides what a graceful failure looks like, because only it knows the channel.

| Channel | Degraded behaviour |
|---|---|
| Consumer app | "I cannot reach the system. Your case is TL-88421 and a human is joining now." |
| Fleet portal | return the last cached adjudication with an explicit staleness banner |
| Ops console | show the raw trace, ops people want the stack |

**The anti pattern:** a `try/except: return "Sorry, something went wrong"` at the top level. That erases the trace, hides the class of failure from your dashboards, and makes the same incident happen next month.

In [ ]:
rule("Degraded answer, channel aware")

def answer_or_degrade(ladder, messages, channel, case_id, **kw):
    r, trace = resilient_call(ladder, messages, **kw)
    if r is not None:
        return r.choices[0].message.content, trace
    last = trace[-1].outcome if trace else "unknown"
    fallback = {
        "consumer": f"I cannot reach the system right now. Your case is {case_id} and a human is joining.",
        "fleet":    f"Live adjudication unavailable ({last}). Showing last cached decision for {case_id}. STALE.",
        "ops":      f"HARD FAIL after {len(trace)} attempts. Terminal action: {last}. Trace attached.",
    }[channel]
    return fallback, trace

for ch in ["consumer", "fleet", "ops"]:
    text, tr = answer_or_degrade(LADDER, MSG, ch, "TL-88421", mock_response="litellm.InternalServerError")
    print(f"  [{ch:8s}] {text}")

### Section 5 recap

| You learned | Why it matters |
|---|---|
| One taxonomy across all providers | your on call engineer learns one vocabulary |
| The four R's: Retry, Re-route, Reshape, Refuse | half of all exceptions must never be retried |
| Specific `except` blocks first | `ContextWindowExceededError` is a `BadRequestError`, `Timeout` is an `APIConnectionError` |
| Backoff without jitter is not backoff | it synchronises your fleet into a thundering herd |
| The terminal state is degrade, not raise | a graceful handoff is a successful outcome |

**Now the pivot.** You just hand built retry, re-route and ladder walking. That was the point: the Router in S6 does exactly this, with capacity awareness you cannot easily hand build. From here on you configure it instead of writing it.

---
# S6. The Dispatcher: `litellm.Router`

**The one idea:** your application stops naming models and starts naming **jobs**.

```
before:  litellm.completion(model="bedrock/us.amazon.nova-micro-v1:0", ...)
after:   router.acompletion(model="tl-fast", ...)
```

`tl-fast` is a **model group**: a logical name backed by N physical deployments. Swapping a model, adding a region, or changing the weights is now a config change with no code change. That single indirection is what turns a demo into a system.

### 6.1 Anatomy of a `model_list`

```mermaid
flowchart LR
    subgraph G["model group: tl-fast"]
      direction TB
      D1["deployment id: micro-use1<br/>model: bedrock/us.amazon.nova-micro-v1:0<br/>region: us-east-1<br/>weight 3, rpm 600"]
      D2["deployment id: micro-usw2<br/>model: bedrock/us.amazon.nova-micro-v1:0<br/>region: us-west-2<br/>weight 1, rpm 200"]
    end
    CALL["router.acompletion(model='tl-fast')"] --> G
```

Three fields, three different jobs, and people mix them up constantly:

| Field | Meaning | Who reads it |
|---|---|---|
| `model_name` | the **logical group**, what your code asks for | your application |
| `litellm_params.model` | the **physical model string** | LiteLLM's provider layer |
| `model_info.id` | the **deployment identity**, stable across restarts | cooldowns, metrics, your logs |

Set `model_info.id` yourself. If you do not, LiteLLM generates a UUID and your cooldown logs become unreadable across restarts.

In [ ]:
# ============================================================
# S6. The Torchlight fleet as a Router. This object is reused for the rest of the notebook.
# ============================================================

def torchlight_model_list(fail: dict[str, str] | None = None):
    """
    Build the fleet. `fail` injects a per-deployment simulated failure, keyed by deployment id.
    Values must be STRINGS (see the trap in 6.4) e.g. 'litellm.InternalServerError'.
    """
    fail = fail or {}

    def lp(model, **extra):
        d = {"model": model}
        d.update(extra)
        return d

    def canned(dep_id, text):
        # deployment level mock: only THIS deployment behaves this way
        if not MOCK:
            return {}
        return {"mock_response": fail.get(dep_id, text)}

    return [
        # tl-fast: two regions, 3:1 weighting toward the primary
        {"model_name": "tl-fast",
         "litellm_params": lp(MICRO, aws_region_name="us-east-1", weight=3, rpm=600, tags=["general"],
                              **canned("micro-use1", "[fast/use1] Case TL-88421: technician EN-14, ETA 22 minutes.")),
         "model_info": {"id": "micro-use1", "max_input_tokens": 128_000}},
        {"model_name": "tl-fast",
         "litellm_params": lp(MICRO, aws_region_name="us-west-2", weight=1, rpm=200, tags=["general"],
                              **canned("micro-usw2", "[fast/usw2] Case TL-88421: technician EN-14, ETA 22 minutes.")),
         "model_info": {"id": "micro-usw2", "max_input_tokens": 128_000}},

        # tl-balanced: 300k context and vision. The long document and photo workhorse.
        {"model_name": "tl-balanced",
         "litellm_params": lp(LITE, aws_region_name="us-east-1", rpm=400, tags=["general", "long-context"],
                              **canned("lite-use1", "[balanced] Case TL-88421: clause 7.2 applies, tow covered, GBP 50 excess.")),
         "model_info": {"id": "lite-use1", "max_input_tokens": 300_000}},

        # tl-deep: reasoning and tool calling
        {"model_name": "tl-deep",
         "litellm_params": lp(HAIKU, aws_region_name="us-east-1", rpm=200, tags=["general"],
                              **canned("haiku-use1", "[deep] Case TL-88421: likely alternator failure, dispatch with a jump pack.")),
         "model_info": {"id": "haiku-use1", "max_input_tokens": 200_000}},

        # tl-guard: safety critical only. Deliberately NOT in the general tag.
        {"model_name": "tl-guard",
         "litellm_params": lp(SONNET, aws_region_name="us-east-1", rpm=60, tags=["safety"],
                              **canned("sonnet-use1", "[guard] Case TL-88421: exit the vehicle, stand behind the barrier. Police notified.")),
         "model_info": {"id": "sonnet-use1", "max_input_tokens": 200_000}},
    ]

def build_router(fail=None, **overrides):
    cfg = dict(
        model_list=torchlight_model_list(fail),
        routing_strategy="simple-shuffle",
        num_retries=2,
        timeout=30,
        cooldown_time=20,
        allowed_fails=2,
        # the three fallback ladders, explained in 6.4
        fallbacks=[{"tl-fast": ["tl-deep"]}, {"tl-deep": ["tl-guard"]}],
        context_window_fallbacks=[{"tl-fast": ["tl-balanced"]}, {"tl-deep": ["tl-balanced"]}],
        content_policy_fallbacks=[{"tl-fast": ["tl-guard"]}],
        enable_pre_call_checks=True,
    )
    cfg.update(overrides)
    return Router(**cfg)

router = build_router()
print("groups:", sorted({d["model_name"] for d in torchlight_model_list()}))
print("deployments:", [d["model_info"]["id"] for d in torchlight_model_list()])

In [ ]:
rule("Your code now names jobs, not models")

async def demo_groups():
    for group in ["tl-fast", "tl-balanced", "tl-deep", "tl-guard"]:
        r = await router.acompletion(model=group, messages=[{"role": "user", "content": "status?"}], max_tokens=80)
        print(f"  {group:12s} served by {r._hidden_params['model_id']:12s} -> {r.choices[0].message.content[:52]}")

await demo_groups()

### 6.2 Load balancing patterns: five strategies, five different problems

The strategy is not a preference. Each one **solves a specific failure** and is wrong for the others.

| Strategy | Picks by | Solves | Fails at | Use it when |
|---|---|---|---|---|
| `simple-shuffle` | random, weighted | spreading load, canary rollouts | a slow deployment still gets its share | default, and for weighted canaries |
| `least-busy` | fewest in flight requests | head of line blocking, uneven request sizes | needs live concurrency, cold start bias | long, variable duration calls (class C) |
| `latency-based-routing` | lowest observed latency | one region degrading | needs warm up traffic, can herd onto one deployment | multi region under partial degradation |
| `cost-based-routing` | cheapest available | spend | ignores quality entirely | homogeneous quality, heterogeneous price |
| `usage-based-routing-v2` | remaining TPM and RPM headroom | throttling before it happens | needs a shared Redis in multi pod | high volume, hard provider quotas |

The **weighted** case deserves emphasis because it is the one people underuse. `weight` on `simple-shuffle` gives you a canary control plane for free: ship a new model at weight 1 against an incumbent at weight 19 and you have a 5% rollout with a one line rollback.

In [ ]:
rule("Measured distribution per strategy (200 calls each)")
import collections

async def distribution(strategy, group="tl-fast", n=200, **kw):
    r = build_router(routing_strategy=strategy, num_retries=0, enable_pre_call_checks=False, **kw)
    c = collections.Counter()
    for _ in range(n):
        res = await r.acompletion(model=group, messages=[{"role": "user", "content": "x"}], max_tokens=20)
        c[res._hidden_params["model_id"]] += 1
    return dict(c)

print("  simple-shuffle on tl-fast (weights 3:1)  ->", await distribution("simple-shuffle"))
print("  least-busy                               ->", await distribution("least-busy", n=60))
print("  latency-based-routing                    ->", await distribution("latency-based-routing", n=60))

# cost-based: give it a group with genuinely different prices
async def cost_demo():
    ml = [
        {"model_name": "any", "litellm_params": {"model": MICRO, **mock("CHEAP")}, "model_info": {"id": "micro"}},
        {"model_name": "any", "litellm_params": {"model": SONNET, **mock("PRICEY")}, "model_info": {"id": "sonnet"}},
    ]
    r = Router(model_list=ml, routing_strategy="cost-based-routing", num_retries=0)
    c = collections.Counter()
    for _ in range(40):
        res = await r.acompletion(model="any", messages=[{"role": "user", "content": "x"}])
        c[res._hidden_params["model_id"]] += 1
    return dict(c)

print("  cost-based-routing (micro vs sonnet)     ->", await cost_demo())
print("\n  Read the first line: 3:1 weights produce a 3:1 split. That is your canary dial.")
print("\n  READ THE SECOND LINE HONESTLY: least-busy sent 100% to one deployment.")
print("  That is not a bug and it is not a recommendation. Mocked calls return instantly,")
print("  so in-flight count is ALWAYS zero, so 'least busy' has no signal and takes the first.")
print("  least-busy needs real concurrency to mean anything. Any benchmark of it on a")
print("  synthetic serial loop is measuring nothing. Same caveat applies to usage-based-v2,")
print("  which needs accumulated TPM/RPM, and to latency-based, which needs warm samples.")

---
### Challenge 6.A

Torchlight runs `tl-fast` across `us-east-1` and `us-west-2`. At 18:00 on a storm day, `us-east-1` does not fail, it just gets **slow**: p95 goes from 400ms to 9 seconds. `us-west-2` stays at 400ms. Requests still succeed, so nothing is in the error budget, but the app feels broken.

| Option | Change |
|---|---|
| **A** | Switch to `least-busy` |
| **B** | Switch to `latency-based-routing` |
| **C** | Keep `simple-shuffle`, drop `timeout` to 2s and let retries re-route |

Commit before scrolling.

### Verdict 6.A

**B is the direct fix. C is the fix you also want. A is a trap here.**

- **A `least-busy`** counts **in flight requests**. A slow deployment holds requests open, so its in flight count goes **up**, so `least-busy` sends it less. That sounds right, and it partly works. The trap: `least-busy` cannot tell "slow" from "handling long requests". On Torchlight's mixed traffic a deployment serving three class C adjudications looks identical to a degraded region. You will de prioritise a healthy node.
- **B `latency-based-routing`** measures exactly the symptom you have. This is what it is for. Caveat: it needs recent samples per deployment, so a cold deployment can be starved. LiteLLM keeps sampling, but the first minute after a restart is noisy.
- **C** is not an alternative, it is the **other half**. `latency-based-routing` shifts the *distribution*; a tight `timeout` bounds the *tail*. Without a timeout, the requests that already went to `us-east-1` still take 9 seconds. The pairing is: latency routing to move traffic, timeout plus retry to rescue the requests already in flight.

**Torchlight's rule:** every model group gets a timeout derived from its **latency budget** (S0.2), never a global default. Class A gets 2s. Class C gets 25s. A global 30s timeout means a status lookup can hang for 30 seconds.

### 6.3 Cooldowns: the circuit breaker you already have

```mermaid
flowchart LR
    H[healthy] -->|allowed_fails exceeded| C[cooling down]
    C -->|cooldown_time elapsed| H
    C -.->|excluded from selection| S[selection pool]
    H --> S
```

| Knob | Meaning | Torchlight setting | Why |
|---|---|---|---|
| `allowed_fails` | failures before a deployment is benched | 2 | one blip is noise, two is a signal |
| `cooldown_time` | seconds on the bench | 20 | long enough to matter, short enough to recover |
| `disable_cooldowns` | turn it all off | never in prod | useful in tests where you want deterministic selection |

**The failure mode nobody predicts:** if every deployment in a group goes into cooldown, the group is empty and the Router raises immediately rather than trying anything. That is correct behaviour, and it is why you need fallbacks to a **different group** (6.4), not just more deployments in the same one.

In [ ]:
rule("Cooldown in action: bench a deployment, watch selection change")

async def cooldown_demo():
    # micro-use1 fails, micro-usw2 is healthy. Same group.
    r = build_router(fail={"micro-use1": "litellm.InternalServerError"},
                     num_retries=0, allowed_fails=1, cooldown_time=30,
                     fallbacks=[], context_window_fallbacks=[], content_policy_fallbacks=[])
    seen = collections.Counter()
    for i in range(8):
        try:
            res = await r.acompletion(model="tl-fast", messages=[{"role": "user", "content": "x"}], max_tokens=20)
            seen[res._hidden_params["model_id"]] += 1
        except Exception as e:
            seen["ERROR:" + type(e).__name__] += 1
    print("  selections over 8 calls:", dict(seen))
    ids = [d["model_info"]["id"] for d in r.get_model_list(model_name="tl-fast")]
    benched = await r.cooldown_cache.async_get_active_cooldowns(model_ids=ids, parent_otel_span=None)
    for dep_id, meta in benched:
        print(f"  benched: {dep_id} for {meta['cooldown_time']}s, status {meta['status_code']}")
    print("\n  Early calls hit the broken deployment and fail. After allowed_fails it is benched,")
    print("  and every later call lands on the healthy region. That is a circuit breaker.")

await cooldown_demo()

### 6.4 Three fallback ladders, and why one is not enough

This is the part that most teams get wrong. LiteLLM has **three separate fallback lists**, because three different failures need three different destinations.

| List | Triggered by | Destination should be | Torchlight ladder |
|---|---|---|---|
| `fallbacks` | any error not covered by the other two | a **different, healthier** group | `tl-fast -> tl-deep -> tl-guard` |
| `context_window_fallbacks` | `ContextWindowExceededError` | a **bigger context** group | `tl-fast -> tl-balanced` (128k to 300k) |
| `content_policy_fallbacks` | `ContentPolicyViolationError` | a **differently aligned** model | `tl-fast -> tl-guard` |

```mermaid
flowchart TB
    REQ["router.acompletion('tl-fast')"] --> TRY[try tl-fast deployments]
    TRY --> ERR{error type}
    ERR -- none --> OK([return])
    ERR -- ContextWindowExceeded --> CWF["context_window_fallbacks<br/>-> tl-balanced 300k"]
    ERR -- ContentPolicyViolation --> CPF["content_policy_fallbacks<br/>-> tl-guard"]
    ERR -- anything else --> GF["fallbacks<br/>-> tl-deep -> tl-guard"]
    CWF --> OK
    CPF --> OK
    GF --> OK
```

**Why routing a context overflow into the generic ladder is a bug:** `tl-fast` and `tl-deep` are 128k and 200k. A prompt that blew 128k might also blow 200k. You would burn a retry, burn a fallback hop, and fail anyway. `context_window_fallbacks` jumps straight to the 300k deployment. One hop, correct destination.

> **Verified trap that will waste an afternoon.** Passing `mock_response=<exception>` at **call time** applies it to **every** attempt, including the fallback, so your fallback demo "fails" and you conclude fallbacks are broken. Inject failures in the **deployment's** `litellm_params` instead. That is what `torchlight_model_list(fail=...)` does. Also note: the `Router` deep copies `model_list`, and LiteLLM exception **instances** are not deep copyable, so per deployment injection must use the **string** form.

In [ ]:
rule("All three fallback ladders, proven")

async def fallback_demos():
    # 1. generic fallback: tl-fast is 500ing -> ladder to tl-deep
    r = build_router(fail={"micro-use1": "litellm.InternalServerError",
                           "micro-usw2": "litellm.InternalServerError"}, num_retries=0)
    res = await r.acompletion(model="tl-fast", messages=[{"role": "user", "content": "fault triage"}], max_tokens=60)
    print(f"  1) generic fallback        tl-fast -> {res._hidden_params['model_id']:12s} {res.choices[0].message.content[:44]}")

    # 2. context window fallback: tl-fast overflows -> jump to 300k tl-balanced
    r = build_router(fail={"micro-use1": "litellm.ContextWindowExceededError",
                           "micro-usw2": "litellm.ContextWindowExceededError"}, num_retries=0)
    res = await r.acompletion(model="tl-fast", messages=[{"role": "user", "content": "60 page contract..."}], max_tokens=60)
    print(f"  2) context_window_fallback tl-fast -> {res._hidden_params['model_id']:12s} {res.choices[0].message.content[:44]}")

await fallback_demos()

**Honesty note on the third ladder.** `content_policy_fallbacks` fires on a real `ContentPolicyViolationError`. LiteLLM's offline injection only raises three exception types from the string form (`RateLimitError`, `ContextWindowExceededError`, `InternalServerError`), and a per deployment exception *instance* cannot be used because the Router deep copies `model_list`. So the third ladder cannot be faked at the deployment level.

It is verifiable at the SDK layer, which is what the cell below does. The routing behaviour is identical to the other two ladders: a separate list, a separate destination, chosen because a refusal needs a **differently aligned** model rather than a bigger or healthier one.

In [ ]:
rule("content_policy: verified at the SDK layer, then mapped to the ladder")

try:
    litellm.completion(model=MICRO, messages=[{"role": "user", "content": "x"}],
                       mock_response=litellm.ContentPolicyViolationError(
                           message="simulated refusal", model=MICRO, llm_provider="bedrock"))
except litellm.ContentPolicyViolationError as e:
    print("  raised          :", type(e).__name__)
    print("  is BadRequest?  :", isinstance(e, litellm.BadRequestError), " <- the S5.2 trap, again")
    print("  ladder consulted: content_policy_fallbacks -> tl-guard")

print("\n  Why a SEPARATE list and not the generic one:")
print("    generic ladder sends a refusal to tl-deep, which may refuse identically.")
print("    A refusal is an ALIGNMENT mismatch, not a capacity or capability problem.")
print("    The destination must differ in policy, not in size.")

In [ ]:
rule("The trap: call level mock_response leaks into the fallback")

async def leak_demo():
    r = build_router(num_retries=0)   # nothing is broken in the fleet
    try:
        await r.acompletion(model="tl-fast", messages=[{"role": "user", "content": "x"}],
                            mock_response="litellm.InternalServerError")   # <-- CALL level
        print("  unexpected success")
    except Exception as e:
        print("  call level injection ->", type(e).__name__)
        print("  The fallback group ALSO received mock_response, so it failed too.")
        print("  Conclusion people wrongly reach: 'fallbacks do not work'.")

    r2 = build_router(fail={"micro-use1": "litellm.InternalServerError",
                            "micro-usw2": "litellm.InternalServerError"}, num_retries=0)
    res = await r2.acompletion(model="tl-fast", messages=[{"role": "user", "content": "x"}])
    print("  deployment level injection ->", res._hidden_params["model_id"], "(fallback worked)")

await leak_demo()

### 6.5 Two router features that do routing work for you

Before you write a single line of classifier code (S7), the Router already does two kinds of intelligent routing.

**`enable_pre_call_checks=True`** filters out deployments whose context window cannot hold the request, **before** dispatching. This is context based routing with no code.

**`enable_tag_filtering=True`** routes on request metadata. Tag a deployment `["eu-resident"]` and pass `metadata={"tags": ["eu-resident"]}`, and only that deployment is eligible. This is how data residency, tenant isolation and per team model access get enforced at the transport layer instead of in fifteen places in your application.

In [ ]:
rule("Pre-call context check: the Router filters on context window before dispatch")

async def precall_demo():
    ml = [
        {"model_name": "auto", "litellm_params": {"model": MICRO, **mock("SERVED BY 128k")},
         "model_info": {"id": "micro-128k", "max_input_tokens": 128_000}},
        {"model_name": "auto", "litellm_params": {"model": LITE, **mock("SERVED BY 300k")},
         "model_info": {"id": "lite-300k", "max_input_tokens": 300_000}},
    ]
    r = Router(model_list=ml, enable_pre_call_checks=True, num_retries=0)
    small = [{"role": "user", "content": "short question"}]
    big   = [{"role": "user", "content": "word " * 70_000}]
    a = await r.acompletion(model="auto", messages=small)
    b = await r.acompletion(model="auto", messages=big)
    print("  short prompt ->", a._hidden_params["model_id"])
    print("  huge prompt  ->", b._hidden_params["model_id"], " <- 128k deployment was filtered out, not tried and failed")

await precall_demo()

In [ ]:
rule("Tag routing: data residency and tenant isolation at the transport layer")

async def tag_demo():
    ml = [
        {"model_name": "adjudicate", "litellm_params": {"model": MICRO, "tags": ["eu-resident"], **mock("EU deployment")},
         "model_info": {"id": "eu-deploy"}},
        {"model_name": "adjudicate", "litellm_params": {"model": LITE, "tags": ["us-only"], **mock("US deployment")},
         "model_info": {"id": "us-deploy"}},
    ]
    r = Router(model_list=ml, enable_tag_filtering=True, num_retries=0)
    for tag in ["eu-resident", "us-only"]:
        res = await r.acompletion(model="adjudicate", messages=[{"role": "user", "content": "x"}],
                                  metadata={"tags": [tag]})
        print(f"  tags={tag:12s} -> {res._hidden_params['model_id']:10s} {res.choices[0].message.content}")
    print("\n  A German fleet customer's contract never touches a US deployment, and the rule")
    print("  lives in ONE place instead of every call site.")

await tag_demo()

---
### Challenge 6.B

Torchlight is adding `tl-deep-v2` (a newer reasoning model). Product wants 5% of class B traffic on it for two weeks, with an instant rollback and no code deploy for the rollback.

| Option | Design |
|---|---|
| **A** | `if random.random() < 0.05: model = "tl-deep-v2"` in the application |
| **B** | Add the new deployment into the **existing `tl-deep` group** with `weight=1` against the incumbent's `weight=19` |
| **C** | Create a separate `tl-deep-v2` group and a feature flag service to pick between groups |

### Verdict 6.B

**B.**

- **A** puts a rollout decision inside application code. Rollback is a deploy. And your logs now have two paths that both say "tl-deep", so you cannot compare them.
- **C** works and is what you build if the two models need **different prompts, different tools or different timeouts**. It is more machinery than a weight change needs.
- **B** is a config change. Rollback is setting `weight=0` or deleting a list entry. And because both deployments carry distinct `model_info.id`, every metric you already collect is automatically split by variant. You get the A/B analysis for free from the field you were told to set in 6.1.

**The general principle, worth writing down:** *anything that is a traffic decision belongs in the Router. Anything that is a behaviour decision belongs in the application.* A canary is traffic. A different prompt is behaviour.

### Section 6 recap

| Concept | Setting |
|---|---|
| Name jobs, not models | `model_name` as a group, code calls the group |
| Deployment identity | always set `model_info.id`, cooldowns and metrics depend on it |
| Load balancing | five strategies, five different failures, pick by symptom |
| Canary rollout | `weight` on `simple-shuffle`, rollback is a config edit |
| Circuit breaker | `allowed_fails` plus `cooldown_time` |
| Three fallback ladders | generic, context window, content policy, three destinations |
| Free intelligent routing | `enable_pre_call_checks`, `enable_tag_filtering` |
| Testing trap | inject failures per **deployment**, string form only |

---
# S7. The Decision Layer: intelligent routing

S6 answered "which **deployment** of a group". S7 answers the harder question: **"which group in the first place?"**

The Router does not answer this. It cannot. Only your domain knows that "smoke from the bonnet with kids in the car" is class D and "where's my van" is class A.

### 7.1 The routing ladder

Six levels. Each one costs more and knows more. **The mistake is starting at L4.**

| L | Name | Decides from | Added latency | Added cost | Explainable | Breaks when |
|---|---|---|---|---|---|---|
| **L0** | Static | nothing, one model | 0 | 0 | totally | traffic is not homogeneous |
| **L1** | Heuristic | keywords, regex, channel, source | under 1ms | 0 | totally | phrasing varies, other languages |
| **L2** | Context | token count, has image, needs tools | under 1ms | 0 | totally | it only sees shape, not meaning |
| **L3** | Intent classifier | a small model returning a label | 150 to 400ms | ~0.00002 | mostly | out of distribution phrasing |
| **L4** | LLM router | a model reasoning about the route | 400ms to 2s | ~0.0004 | partly | non determinism, prompt drift, cost |
| **L5** | Cascade | answer cheap, verify, escalate on doubt | varies | varies | fully traceable | needs a good confidence signal |

```mermaid
flowchart TB
    IN([request]) --> L1{L1 heuristic<br/>hard rules and safety keywords}
    L1 -- hard match --> OUT([route decided])
    L1 -- no match --> L2{L2 context<br/>tokens, image, tools}
    L2 -- constraint binds --> OUT
    L2 -- free --> L3{L3 intent classifier<br/>small model, structured label}
    L3 -- confidence high --> OUT
    L3 -- confidence low --> L4{L4 LLM router<br/>reasons about the route}
    L4 --> OUT
    OUT --> ANS[answer on chosen group]
    ANS --> L5{L5 verify<br/>did the cheap tier actually do it?}
    L5 -- ok --> DONE([respond])
    L5 -- doubt --> ESC[escalate one tier and re-answer]
    ESC --> DONE
```

**Two rules that save you from the common disaster:**

1. **L1 and L2 are constraints, not preferences.** They run first and they are allowed to *end* the decision. A safety keyword must never be overruled by a classifier's opinion.
2. **Every level down is a fallback for the level above.** If the classifier times out, you land on the heuristic's answer, not on an exception.

---
### Challenge 7.A

Torchlight's first routing design was: "send everything to a small model, ask it to output a JSON label, route on the label." It worked in the demo and failed in production three ways in the first week.

Predict the three failures before scrolling. Then pick what you would fix first.

| Option | First fix |
|---|---|
| **A** | Better prompt for the classifier |
| **B** | Put hard rules in front of the classifier |
| **C** | Fine tune a classifier on Torchlight's own traffic |

### Verdict 7.A

The three production failures, in the order they happened:

1. **The classifier returned prose.** `"Sure! The intent here is: safety_critical"` is not JSON. `json.loads` raised, the exception handler routed to the default, and the default was `tl-fast`. **A safety critical call was routed to the cheapest model by an exception handler.**
2. **The classifier was itself throttled.** At peak, the routing call hit `RateLimitError`, and the routing layer had no fallback because "it's just a classifier". Every request failed at the routing step, before any answer was attempted.
3. **Latency doubled on class A.** 55% of traffic is a sub second status lookup. Adding a 300ms classifier in front of a 400ms answer is a 75% latency regression on the majority of traffic, to make a decision that a keyword match answers correctly.

**First fix: B.** Not because prompting does not help, but because A and C both leave the *architecture* wrong. A classifier with a better prompt still fails open on parse errors and still adds latency to the 55%. Hard rules in front give you: safety keywords never reach the classifier, and the cheap obvious cases never pay the classifier tax.

**The general principle:** *route with the cheapest mechanism that is sufficient, and let each layer fail into the layer above it, never into a default.*

Notice the phrase "**fail into the layer above**". Failing to a default is how a safety call reaches Nova Micro.

### 7.2 L1: heuristic routing

The unglamorous layer that handles most of your correctness. Two jobs:

- **Hard safety rules.** These are non negotiable and must be auditable by someone who is not an engineer.
- **Cheap obvious wins.** Channel, source system, and a small keyword set.

In [ ]:
# ============================================================
# L1. Heuristic routing. Rules as DATA so a safety officer can read them.
# ============================================================
import re

@dataclass
class Decision:
    group: str
    reason: str
    level: str
    confidence: float = 1.0
    hard: bool = False          # hard decisions cannot be overridden downstream

SAFETY_TERMS = [
    "motorway", "hard shoulder", "highway", "smoke", "fire", "burning",
    "child", "kids", "baby", "accident", "collision", "injured", "unconscious",
]
FAST_TERMS = ["where is", "eta", "how long", "status", "arrived", "tracking", "reference number"]

SAFETY_RE = re.compile(r"\b(" + "|".join(map(re.escape, SAFETY_TERMS)) + r")\b", re.I)
FAST_RE   = re.compile(r"\b(" + "|".join(map(re.escape, FAST_TERMS)) + r")\b", re.I)

def l1_heuristic(text: str, channel: str = "consumer") -> Optional[Decision]:
    hits = SAFETY_RE.findall(text)
    if hits:
        return Decision("tl-guard", f"safety terms {sorted(set(h.lower() for h in hits))}", "L1", 1.0, hard=True)
    if channel == "ops":
        return Decision("tl-deep", "internal ops console, no cost sensitivity", "L1", 1.0)
    if FAST_RE.search(text) and len(text) < 200:
        return Decision("tl-fast", "short status lookup pattern", "L1", 0.9)
    return None

rule("L1 heuristic on real Torchlight phrasings")
SAMPLES = [
    ("Where is my technician for TL-88421?", "consumer"),
    ("Broken down on the M1 hard shoulder with two kids in the car", "consumer"),
    ("Grinding noise from the front left wheel, getting louder", "consumer"),
    ("Adjudicate FL-2291 against the attached master services agreement", "fleet"),
]
for text, ch in SAMPLES:
    d = l1_heuristic(text, ch)
    print(f"  {text[:52]:54s} -> {d.group if d else 'no match, escalate to L2':22s} {d.reason if d else ''}")

### 7.3 L2: context and capability routing

L2 looks at the **shape** of the request, never its meaning. Three inputs, all free:

| Input | Source | Routing consequence |
|---|---|---|
| token count | `litellm.token_counter` | overflow the group's `max_input_tokens`, go to the 300k group |
| has an image | inspect the message content blocks | must go to a `supports_vision` group |
| needs tools | your own agent config | must go to a `supports_function_calling` group |

These are **hard constraints**. They can only shrink the eligible set, never expand it. A request with an image cannot be served by a text only group no matter what the classifier thinks.

In [ ]:
# ============================================================
# L2. Context and capability routing. Constraints, not preferences.
# ============================================================
GROUP_CAPS = {
    "tl-fast":     {"ctx": 128_000, "vision": False, "tools": True,  "tier": 0},
    "tl-balanced": {"ctx": 300_000, "vision": True,  "tools": True,  "tier": 1},
    "tl-deep":     {"ctx": 200_000, "vision": True,  "tools": True,  "tier": 2},
    "tl-guard":    {"ctx": 200_000, "vision": True,  "tools": True,  "tier": 3},
}
# Nova Micro has no vision. Verified in S4.4.

def has_image(messages) -> bool:
    for m in messages:
        c = m.get("content")
        if isinstance(c, list) and any(b.get("type") in ("image_url", "image") for b in c):
            return True
    return False

def l2_context(messages, candidate: str, needs_tools: bool = False) -> Decision:
    """Take a candidate group and shrink to the smallest group that satisfies every constraint."""
    ntok  = litellm.token_counter(model=HAIKU, messages=[m for m in messages if isinstance(m.get("content"), str)])
    img   = has_image(messages)
    caps  = GROUP_CAPS[candidate]
    fails = []
    if ntok > caps["ctx"]:      fails.append(f"needs {ntok:,} ctx, group has {caps['ctx']:,}")
    if img and not caps["vision"]: fails.append("needs vision")
    if needs_tools and not caps["tools"]: fails.append("needs tools")
    if not fails:
        return Decision(candidate, f"{ntok:,} tokens, image={img}, fits", "L2")

    for g, c in sorted(GROUP_CAPS.items(), key=lambda kv: kv[1]["tier"]):
        if ntok <= c["ctx"] and (not img or c["vision"]) and (not needs_tools or c["tools"]):
            return Decision(g, f"constraint promote from {candidate}: {'; '.join(fails)}", "L2", hard=True)
    return Decision("tl-balanced", "no group satisfies constraints, largest context wins", "L2", hard=True)

rule("L2 constraints override cheap preferences")
CONTRACT = [{"role": "user", "content": "Adjudicate this contract. " + ("clause " * 60_000)}]
PHOTO = [{"role": "user", "content": [
    {"type": "text", "text": "What does this warning light mean?"},
    {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,AAAA"}}]}]

for label, msgs in [("short text", [{"role": "user", "content": "eta?"}]),
                    ("60k word contract", CONTRACT),
                    ("dashboard photo", PHOTO)]:
    d = l2_context(msgs, candidate="tl-fast")
    print(f"  {label:20s} candidate tl-fast -> {d.group:12s} {d.reason}")

### 7.4 L3: the intent classifier

Now, and only now, do we spend a model call on the routing decision. Three things make the difference between a classifier that works and the one from Challenge 7.A.

1. **Constrain the output.** Ask for a single token label, give the exact allowed set, set `max_tokens` low. A classifier that can write a paragraph will.
2. **Parse defensively.** Never `json.loads` a model output without a repair path and a **safe** default. Safe means "the more expensive tier", never the cheaper one.
3. **Budget it.** Wrap the classifier in its own timeout and its own fallback group. If it fails, you fall back to L1's answer, not to a hardcoded default.

In [ ]:
# ============================================================
# L3. Intent classifier. Small model, constrained output, defensive parse.
# ============================================================
INTENTS = {
    "status":    "tl-fast",      # class A
    "triage":    "tl-deep",      # class B
    "coverage":  "tl-balanced",  # class C
    "safety":    "tl-guard",     # class D
}

CLASSIFIER_PROMPT = (
    "Classify the roadside assistance message into exactly one label.\n"
    "Labels: status | triage | coverage | safety\n"
    "status   = asking where the technician is, ETA, case progress\n"
    "triage   = describing a fault, wants a diagnosis or a dispatch decision\n"
    "coverage = asking who pays, contract terms, entitlement\n"
    "safety   = anyone is in danger, on a live carriageway, fire, injury, children exposed\n"
    "Reply with ONLY the label. No punctuation. No explanation."
)

def parse_label(raw: str) -> tuple[Optional[str], float]:
    """Defensive parse. Returns (label, confidence). Never guesses toward the cheap tier."""
    if not raw:
        return None, 0.0
    t = raw.strip().strip('".` \n').lower()
    if t in INTENTS:
        return t, 1.0                       # clean single label
    found = [k for k in INTENTS if re.search(rf"\b{k}\b", t)]
    if len(found) == 1:
        return found[0], 0.6                # label embedded in prose, lower confidence
    return None, 0.0                        # unparseable

async def l3_intent(router, text: str, timeout_s: float = 1.5, canned: str | None = None):
    try:
        r = await asyncio.wait_for(
            router.acompletion(
                model="tl-fast",
                messages=[{"role": "system", "content": CLASSIFIER_PROMPT},
                          {"role": "user", "content": text}],
                max_tokens=6, temperature=0.0,
                **({"mock_response": canned} if (MOCK and canned is not None) else {}),
            ), timeout=timeout_s)
        label, conf = parse_label(r.choices[0].message.content)
        if label is None:
            return None, "classifier output unparseable"
        return Decision(INTENTS[label], f"intent={label}", "L3", conf), None
    except asyncio.TimeoutError:
        return None, f"classifier exceeded {timeout_s}s budget"
    except Exception as e:
        return None, f"classifier failed: {type(e).__name__}"

rule("L3 classifier, including the three ways it goes wrong")
CASES = [
    ("Where is my van?",                          "status"),
    ("Engine is making a grinding noise",         "triage"),
    ("Sure! The intent here is: safety",          "prose wrapper, still parseable"),
    ("I think it might be a coverage or a safety question", "ambiguous, two labels"),
    ("Absolutely, happy to help with that!",      "no label at all"),
]
for text, canned in CASES:
    d, err = await l3_intent(router, text, canned=canned)
    if d:
        print(f"  {canned[:38]:40s} -> {d.group:12s} conf={d.confidence:.1f}")
    else:
        print(f"  {canned[:38]:40s} -> NO DECISION   ({err})")

In [ ]:
rule("The failure that matters: the classifier itself is throttled")

async def classifier_down():
    broken = build_router(fail={"micro-use1": "litellm.RateLimitError",
                                "micro-usw2": "litellm.RateLimitError"},
                          num_retries=0, fallbacks=[], context_window_fallbacks=[], content_policy_fallbacks=[])
    d, err = await l3_intent(broken, "grinding noise from the wheel")
    print("  classifier result:", d, "|", err)
    print("\n  WRONG recovery: route to a hardcoded default, which is usually the cheap group.")
    print("  RIGHT recovery: fall back to the L1 heuristic decision, and if L1 had no match,")
    print("                  route UP a tier, not down. Uncertainty is a reason to spend more, not less.")

await classifier_down()

### 7.5 L4: the LLM router

Use L4 when the routing decision itself needs **judgement**, not classification. Torchlight's real L4 case: a fleet manager writes three paragraphs mixing a fault description, a contract question and a complaint. There is no single label. Something has to decide what the request *primarily* is, and whether it should be **split**.

**The design that makes L4 safe:**

| Guard | Why |
|---|---|
| Structured output with an explicit schema | you get a route, not an essay |
| The router model is never the answering model | a router that can answer will answer, and you lose the cost control |
| A confidence field, and a rule for low confidence | route up on doubt |
| Its own timeout and fallback ladder | it is a model call, it fails like one |
| Log the reason string | this is your only debugging handle for a non deterministic decision |

In [ ]:
# ============================================================
# L4. LLM router. Judgement, structured, never the answering model.
# ============================================================
ROUTER_SCHEMA_PROMPT = """You are a ROUTER, not an assistant. You never answer the user.
Given a roadside assistance message, output ONLY minified JSON:
{"group":"tl-fast|tl-balanced|tl-deep|tl-guard","confidence":0.0-1.0,"reason":"<12 words","split":["..."]}

tl-fast     cheap, fast, factual lookups
tl-balanced 300k context and vision, long documents and photos
tl-deep     reasoning, diagnosis, tool use
tl-guard    anything where a person could be harmed

If the message contains several distinct asks, list them in "split"."""

def safe_json(raw: str) -> Optional[dict]:
    """Model JSON is not JSON. Repair the two common breakages, then give up honestly."""
    if not raw:
        return None
    t = raw.strip()
    t = re.sub(r"^```(?:json)?|```$", "", t, flags=re.M).strip()   # fenced blocks
    m = re.search(r"\{.*\}", t, re.S)                              # prose around the object
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

async def l4_llm_router(router, text: str, canned: str | None = None):
    try:
        r = await router.acompletion(
            model="tl-deep",                      # judgement needs a real model
            messages=[{"role": "system", "content": ROUTER_SCHEMA_PROMPT},
                      {"role": "user", "content": text}],
            max_tokens=160, temperature=0.0,
            **({"mock_response": canned} if (MOCK and canned is not None) else {}),
        )
        obj = safe_json(r.choices[0].message.content)
        if not obj or obj.get("group") not in GROUP_CAPS:
            return None, "router output invalid"
        return Decision(obj["group"], obj.get("reason", ""), "L4", float(obj.get("confidence", 0.5))), obj.get("split", [])
    except Exception as e:
        return None, f"router failed: {type(e).__name__}"

rule("L4 on a message no single label fits")
MESSY = ("My van's turbo is whining badly and it cut out twice on the A34. "
         "Also I need to know if the tow is covered under FL-2291 because last time "
         "you charged us and the contract says otherwise. And your app is terrible.")

canned = json.dumps({"group": "tl-deep", "confidence": 0.72,
                     "reason": "primary ask is a mechanical diagnosis",
                     "split": ["turbo fault diagnosis", "coverage check FL-2291", "complaint"]})
d, split = await l4_llm_router(router, MESSY, canned=canned)
print("  route     :", d.group, f"(confidence {d.confidence})")
print("  reason    :", d.reason)
print("  split into:", split)
print("\n  The 'split' field is the real value of L4. A classifier gives you one label for a")
print("  three part message and silently drops two thirds of what the customer asked.")

In [ ]:
rule("L4 fails differently: valid text, invalid decision")
for bad in ['I think tl-deep is best!', '```json\n{"group":"tl-turbo"}\n```', '{"group": "tl-fast"']:
    d, err = await l4_llm_router(router, "x", canned=bad)
    print(f"  {bad[:40]:42s} -> {d.group if d else 'REJECTED'} {'' if d else '(' + str(err) + ')'}")
print("\n  Note the middle case: valid JSON, unknown group. Schema validation is not optional.")
print("  Without the `obj['group'] not in GROUP_CAPS` check you would KeyError in production.")

### 7.6 L5: the cascade with escalation

The highest leverage pattern in the whole notebook, and the one that actually moves the cost curve.

**Idea:** do not try to predict which requests need the expensive model. **Answer cheaply, check the answer, escalate only the ones that failed the check.**

```mermaid
flowchart TB
    Q([request]) --> R1[answer on tl-fast]
    R1 --> V{verify}
    V -- passes --> DONE([respond, cheap])
    V -- fails --> R2[re-answer on tl-deep]
    R2 --> V2{verify}
    V2 -- passes --> DONE
    V2 -- fails --> R3[tl-guard + human flag]
    R3 --> DONE
```

**The verifier is the whole design.** Three kinds, in increasing cost:

| Verifier | Cost | Catches |
|---|---|---|
| Structural: did it produce the required fields, a valid case id, a number in range? | free | format failures, hallucinated ids |
| Self reported confidence: ask the cheap model to also emit a confidence | ~0 | the model's own uncertainty, weakly calibrated |
| Judge model: a second cheap call scoring the answer | ~1 extra cheap call | reasoning failures |

**Economics, with the actual formula rather than a vibe.**

$$\text{cascade} = c_{\text{cheap}} + r \cdot c_{\text{deep}} \qquad \text{always-deep} = c_{\text{deep}}$$

Setting them equal gives the break even escalation rate:

$$r^{*} = 1 - \frac{c_{\text{cheap}}}{c_{\text{deep}}}$$

**The break even depends entirely on the price ratio between your two tiers, not on some universal 30 or 40 percent rule of thumb.** Nova Micro against Haiku 4.5 is roughly a 33x ratio, so $r^{*}$ is about 97%: the cascade stays cheaper almost no matter how often it escalates. Put Haiku 4.5 against Sonnet 4.5 instead, a ratio near 3x, and $r^{*}$ drops to about 67%, which is a real constraint.

**So the thing to actually monitor is not cost, it is latency.** Every escalated request pays **two full round trips plus the verifier**. At a 20% escalation rate your p95 is set by the escalated path, not the cheap one. A cascade that saves 90% of spend and doubles p95 on class D is a bad trade, and the cost dashboard will not tell you that.

The cell below prints the table for your own tier pair. Run it before you commit to a cascade.

In [ ]:
# ============================================================
# L5. Cascade with verify-then-escalate, and the economics.
# ============================================================
TIERS = ["tl-fast", "tl-deep", "tl-guard"]

def structural_verify(text: str, require_case: bool = True) -> tuple[bool, str]:
    if not text or len(text.strip()) < 15:
        return False, "answer too short to be a real answer"
    if require_case and not re.search(r"\bTL-\d{5}\b", text):
        return False, "no valid case reference in the answer"
    if re.search(r"\b(i (can't|cannot)|not sure|unable to determine)\b", text, re.I):
        return False, "model self-reported inability"
    return True, "structural checks passed"

async def cascade(router, messages, start="tl-fast", verifier=structural_verify, canned_by_tier=None):
    canned_by_tier = canned_by_tier or {}
    trail = []
    for tier in TIERS[TIERS.index(start):]:
        r = await router.acompletion(model=tier, messages=messages, max_tokens=200,
                                     **({"mock_response": canned_by_tier[tier]}
                                        if (MOCK and tier in canned_by_tier) else {}))
        text = r.choices[0].message.content
        ok, why = verifier(text)
        trail.append((tier, "PASS" if ok else "ESCALATE", why))
        if ok:
            return text, trail
    return text, trail + [("human", "HANDOFF", "all tiers failed verification")]

rule("Cascade: cheap answer passes")
msgs = [{"role": "user", "content": "Status of TL-88421?"}]
ans, trail = await cascade(router, msgs, canned_by_tier={
    "tl-fast": "Case TL-88421: technician EN-14 assigned, ETA 22 minutes, currently on the A34."})
for t in trail: print(f"    {t[0]:10s} {t[1]:10s} {t[2]}")
print("   answer:", ans[:70])

rule("Cascade: cheap answer fails structurally, escalates once")
ans, trail = await cascade(router, msgs, canned_by_tier={
    "tl-fast": "I'm not sure, please hold.",
    "tl-deep": "Case TL-88421: alternator suspected. Technician EN-14 dispatched with a jump pack, ETA 22 minutes."})
for t in trail: print(f"    {t[0]:10s} {t[1]:10s} {t[2]}")
print("   answer:", ans[:80])

In [ ]:
rule("Cascade economics: where the break-even sits")

def tier_cost(model, tin=800, tout=300):
    i = litellm.get_model_info(model)
    return tin * i["input_cost_per_token"] + tout * i["output_cost_per_token"]

c_fast, c_deep = tier_cost(MICRO), tier_cost(HAIKU)
print(f"  cost per call  tl-fast ${c_fast:.6f}   tl-deep ${c_deep:.6f}   ratio {c_deep/c_fast:.0f}x\n")
print(f"  {'escalation rate':>16s} {'cascade cost':>14s} {'always-deep':>13s} {'saving':>9s}")
for rate in [0.05, 0.10, 0.20, 0.40, 0.60, 0.90]:
    casc = c_fast + rate * c_deep
    print(f"  {rate:>15.0%} {casc:>14.6f} {c_deep:>13.6f} {1 - casc/c_deep:>8.0%}")

print("\n  break-even r* = 1 - (cheap / expensive), for each tier pair you might choose:")
for a, b, la, lb in [(MICRO, HAIKU, "micro", "haiku4.5"),
                     (MICRO, SONNET, "micro", "sonnet4.5"),
                     (HAIKU, SONNET, "haiku4.5", "sonnet4.5"),
                     (LITE, HAIKU, "nova-lite", "haiku4.5")]:
    ca, cb = tier_cost(a), tier_cost(b)
    print(f"    {la:10s} -> {lb:10s} ratio {cb/ca:5.1f}x   r* = {1 - ca/cb:.0%}")

print("\n  The rule of thumb people repeat ('cascades stop paying around 30-40%') is only")
print("  true for tier pairs about 1.5x apart. Compute r* for YOUR pair.")
print("\n  And the metric that actually bites: LATENCY. An escalated request pays two full")
print("  round trips plus the verifier. At 20% escalation your p95 is the escalated path.")

### 7.7 Wiring the ladder into one router

Now assemble L1 through L5 into the object the rest of the notebook uses. Read the order carefully: **hard constraints win, then heuristics, then paid intelligence, and every failure escalates upward.**

In [ ]:
# ============================================================
# The Torchlight decision layer. L1 -> L2 -> L3 -> L4, fail upward.
# ============================================================
TIER_ORDER = ["tl-fast", "tl-balanced", "tl-deep", "tl-guard"]

def promote(group: str) -> str:
    i = TIER_ORDER.index(group)
    return TIER_ORDER[min(i + 1, len(TIER_ORDER) - 1)]

class TorchlightRouter:
    def __init__(self, router, confidence_floor=0.75):
        self.router, self.floor = router, confidence_floor
        self.audit: list[dict] = []

    async def decide(self, messages, channel="consumer", needs_tools=False, canned=None) -> Decision:
        canned = canned or {}
        text = next((m["content"] for m in messages
                     if m["role"] == "user" and isinstance(m["content"], str)), "")
        steps = []

        # L1: hard safety rules, and cheap obvious wins
        d = l1_heuristic(text, channel)
        if d and d.hard:
            steps.append("L1 hard match")
            return self._log(d, steps, messages, needs_tools)

        # L3: paid classification, only if L1 did not settle it
        if d is None:
            d3, err = await l3_intent(self.router, text, canned=canned.get("l3"))
            if d3 and d3.confidence >= self.floor:
                d, _ = d3, steps.append("L3 confident")
            else:
                steps.append(f"L3 unusable ({err or 'low confidence'})")
                # L4: judgement, only when L3 could not settle it
                d4, extra = await l4_llm_router(self.router, text, canned=canned.get("l4"))
                if d4 and d4.confidence >= self.floor:
                    d, _ = d4, steps.append("L4 confident")
                elif d4:
                    d = Decision(promote(d4.group), f"L4 low confidence {d4.confidence}, promoted", "L4+", d4.confidence)
                    steps.append("L4 low confidence, promoted one tier")
                else:
                    d = Decision("tl-deep", "no layer could decide, route UP not down", "fallback")
                    steps.append("all layers failed, safe-up default")
        else:
            steps.append("L1 soft match")

        return self._log(d, steps, messages, needs_tools)

    def _log(self, d, steps, messages, needs_tools):
        # L2 always runs last as a CONSTRAINT. l2_context only returns a DIFFERENT group
        # when the candidate genuinely failed a hard constraint, so any change is accepted.
        # Do NOT gate this on a tier comparison: tl-balanced is cheaper than tl-deep but
        # has a LARGER context window, so a tier check would block a correct promotion.
        c = l2_context(messages, d.group, needs_tools)
        if c.group != d.group:
            steps.append(f"L2 constraint promoted {d.group} -> {c.group}: {c.reason}")
            d = Decision(c.group, c.reason, d.level + "+L2", d.confidence, hard=True)
        self.audit.append({"group": d.group, "level": d.level, "reason": d.reason, "steps": steps})
        return d

tr = TorchlightRouter(router)
print("decision layer ready")

In [ ]:
rule("The full ladder on Torchlight traffic")

SCENARIOS = [
    ("Where is my technician for TL-88421?", "consumer", {}, {}),
    ("Stranded on the M1 hard shoulder, two kids in the car", "consumer", {}, {}),
    ("Grinding noise from the front wheel, worse when braking", "consumer", {}, {"l3": "triage"}),
    ("Absolutely, happy to help!", "consumer", {}, {"l3": "garbage",
        "l4": json.dumps({"group": "tl-fast", "confidence": 0.4, "reason": "unclear ask"})}),
]
for text, ch, _, canned in SCENARIOS:
    d = await tr.decide([{"role": "user", "content": text}], channel=ch, canned=canned)
    print(f"\n  {text[:56]}")
    print(f"    -> {d.group}  [{d.level}]  {d.reason}")
    for s in tr.audit[-1]["steps"]:
        print(f"       . {s}")

# and the constraint case: a photo forces a vision group regardless of intent
d = await tr.decide(PHOTO, canned={"l3": "status"})
print(f"\n  [dashboard photo, classifier said 'status']")
print(f"    -> {d.group}  [{d.level}]  {d.reason}")
print("       L2 overrode a cheap intent because Nova Micro has no vision. Constraints beat preferences.")

### Section 7 recap

| Level | Build it when |
|---|---|
| **L1 heuristic** | always. safety rules must never depend on a model |
| **L2 context** | always. hard constraints, run last, can only promote |
| **L3 classifier** | when phrasing varies more than a keyword list can hold |
| **L4 LLM router** | when the decision needs judgement or the request needs splitting |
| **L5 cascade** | when the escalation rate is measurably below the break even point |

**The three rules to carry out of this section:**

1. Route with the cheapest sufficient mechanism.
2. Every layer fails **into the layer above**, never into a default.
3. Uncertainty routes **up** a tier. Cost optimisation is never the tie breaker on a safety path.

---
# S8. Plugins: the hooks that turn LiteLLM into a control plane

Everything so far decided **where** a call goes. Plugins decide **what happens around it**: redaction, cost accounting, budget enforcement, residency filtering, tracing.

LiteLLM's plugin surface is one class: `litellm.integrations.custom_logger.CustomLogger`. You subclass it, override the hooks you care about, and register the instance in `litellm.callbacks`. It then fires for **both** direct SDK calls and Router calls, which is exactly what you want.

### 8.1 The hook map

These are the hooks worth knowing. All verified against `litellm 1.95.0`.

| Hook | Fires | Torchlight use |
|---|---|---|
| `log_pre_api_call(model, messages, kwargs)` | before the provider call | redaction, prompt audit |
| `async_log_success_event(kwargs, response_obj, start_time, end_time)` | on success | cost ledger, latency histogram |
| `async_log_failure_event(kwargs, response_obj, start_time, end_time)` | on failure | error class counters, alerting |
| `log_success_fallback_event(original_model_group, kwargs, original_exception)` | when a **fallback rescued** the request | the single most valuable signal you are not collecting |
| `log_failure_fallback_event(...)` | when the fallback also failed | page someone |
| `async_filter_deployments(model, healthy_deployments, messages, request_kwargs, ...)` | inside the Router, before selection | residency, tenant isolation, custom eligibility |

```mermaid
flowchart TB
    REQ([router.acompletion]) --> F["async_filter_deployments<br/>shrink the eligible set"]
    F --> P["log_pre_api_call<br/>redact and audit"]
    P --> CALL[provider call]
    CALL -- ok --> S["async_log_success_event<br/>cost, latency, tokens"]
    CALL -- error --> FE["async_log_failure_event<br/>error class counter"]
    FE --> FB{fallback?}
    FB -- rescued --> SF["log_success_fallback_event<br/>SILENT DEGRADATION SIGNAL"]
    FB -- also failed --> FF["log_failure_fallback_event<br/>page"]
    SF --> S
```

**Why `log_success_fallback_event` deserves its own callout.** When a fallback works, your users see success, your error rate stays flat, and your dashboards look green **while your primary model group is down**. This is the definition of silent degradation. Without this hook, Torchlight discovers its `us-east-1` outage a week later on the invoice, because everything quietly went to the more expensive `tl-guard`.

---
### Challenge 8.A

Finance asks for a hard cap: no team may exceed its monthly model budget. An engineer proposes `litellm.max_budget = 500`.

| Option | Approach |
|---|---|
| **A** | `litellm.max_budget = 500`, it is built in |
| **B** | A `CustomLogger` that accumulates `response_cost` and raises once the ceiling is crossed |
| **C** | Do it at the proxy with per key budgets, not in the SDK at all |

### Verdict 8.A

**C is the production answer. B is the answer inside one process. A does not work.**

**A does not work, and this is a verified finding, not an opinion.** In `litellm 1.95.0` the budget check reads `litellm._current_cost`, and that counter is not incremented by ordinary `completion` calls. The cell below runs six calls against a ceiling of one hundred millionth of a dollar and the counter stays at `0.0`. Setting `max_budget` gives you the *feeling* of a cap with none of the behaviour, which is worse than no cap at all.

**B works and has a hard limit.** Your process only knows what your process spent. Three pods means three independent counters and three times the budget. Fine for a single worker or a batch job. Not a company control.

**C is the real answer** because a budget is an **organisational** fact, not a process fact. It has to live somewhere all callers pass through, which is the definition of the Terminal hat (S9). The SDK plugin in B is still worth building, as a **local guard rail and a cost attribution feed**, which is what we do below.

In [ ]:
rule("Verifying that litellm.max_budget does not enforce")

litellm.max_budget = 1e-8          # absurdly low ceiling
litellm._current_cost = 0.0
for i in range(6):
    try:
        litellm.completion(model=MICRO, messages=[{"role": "user", "content": "x"}], **mock("y"))
        print(f"  call {i}: succeeded, litellm._current_cost = {litellm._current_cost}")
    except litellm.BudgetExceededError as e:
        print(f"  call {i}: BudgetExceededError"); break
litellm.max_budget = 0.0            # reset, do not leave this set

print("\n  Six calls, ceiling of $0.00000001, counter never moved. Build the guard yourself.")

In [ ]:
# ============================================================
# S8. Three real Torchlight plugins.
# ============================================================
from litellm.integrations.custom_logger import CustomLogger

PII_PATTERNS = [
    (re.compile(r"\b\d{2}\s?\d{2}\s?\d{2}\b"), "[SORT-CODE]"),
    (re.compile(r"\b[A-Z]{2}\d{2}\s?[A-Z]{3}\b"), "[VRM]"),          # UK plate
    (re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b"), "[EMAIL]"),
    (re.compile(r"\b(?:\+44|0)7\d{9}\b"), "[MOBILE]"),
]

def redact(text: str) -> tuple[str, list[str]]:
    hits = []
    for rx, tag in PII_PATTERNS:
        text, n = rx.subn(tag, text)
        if n:
            hits.append(f"{tag}x{n}")
    return text, hits


class TorchlightPlugin(CustomLogger):
    """One plugin, three jobs: redact, account, and detect silent degradation."""

    def __init__(self, monthly_budget_usd: float):
        super().__init__()
        self.budget = monthly_budget_usd
        self.spend = 0.0
        self.by_group: dict[str, float] = {}
        self.errors: dict[str, int] = {}
        self.silent_degradations: list[dict] = []
        self.redactions: list[str] = []
        self.brake_engaged = False

    # ---- 1. redaction, before the provider ever sees the prompt ----
    def log_pre_api_call(self, model, messages, kwargs):
        for m in messages or []:
            if isinstance(m.get("content"), str):
                clean, hits = redact(m["content"])
                if hits:
                    m["content"] = clean          # mutated in place, before dispatch
                    self.redactions.extend(hits)

    # ---- 2. cost ledger and budget brake ----
    async def async_log_success_event(self, kwargs, response_obj, start_time, end_time):
        cost = kwargs.get("response_cost") or 0.0
        group = ((kwargs.get("litellm_params") or {}).get("metadata") or {}).get("model_group", "direct")
        self.spend += cost
        self.by_group[group] = self.by_group.get(group, 0.0) + cost
        if self.spend > self.budget and not self.brake_engaged:
            self.brake_engaged = True

    async def async_log_failure_event(self, kwargs, response_obj, start_time, end_time):
        name = type(kwargs.get("exception")).__name__
        self.errors[name] = self.errors.get(name, 0) + 1

    # ---- 3. silent degradation detector ----
    def log_success_fallback_event(self, original_model_group, kwargs, original_exception):
        self.silent_degradations.append({
            "group": original_model_group,
            "cause": type(original_exception).__name__,
            "at": time.strftime("%H:%M:%S"),
        })

plugin = TorchlightPlugin(monthly_budget_usd=0.0015)  # tiny on purpose, so the brake trips in this demo
litellm.callbacks = [plugin]
print("plugin registered on litellm.callbacks (fires for SDK and Router calls)")

In [ ]:
rule("Plugin end to end: redaction, ledger, degradation detection")

async def plugin_demo():
    # tl-fast is down in both regions, so every call is rescued by the fallback ladder
    r = build_router(fail={"micro-use1": "litellm.InternalServerError",
                           "micro-usw2": "litellm.InternalServerError"}, num_retries=0)

    dirty = ("Caller mike.ellis@fleetco.co.uk on 07700900123, van AB12 CDE, "
             "account sort 40 21 09. Broken down, needs a tow.")
    print("  BEFORE:", dirty)

    res = await r.acompletion(model="tl-fast",
                              messages=[{"role": "user", "content": dirty}], max_tokens=60)
    await asyncio.sleep(0.4)                       # async callbacks are fire and forget

    print("  redacted tags   :", plugin.redactions)
    print("  served by       :", res._hidden_params["model_id"], "(NOT tl-fast)")
    print("  error counters  :", plugin.errors)
    print("  SILENT DEGRADE  :", plugin.silent_degradations)
    print("\n  The user got a correct answer. The error rate a naive dashboard shows is ZERO.")
    print("  Only log_success_fallback_event knows tl-fast is on the floor.")

await plugin_demo()

In [ ]:
rule("Cost ledger and the budget brake")

async def ledger_demo():
    r = build_router(num_retries=0)
    for i in range(30):
        await r.acompletion(model="tl-deep",
                            messages=[{"role": "user", "content": f"triage case {i}"}], max_tokens=80)
    await asyncio.sleep(0.6)
    print(f"  total spend    : ${plugin.spend:.6f}")
    print(f"  budget         : ${plugin.budget:.6f}")
    print(f"  brake engaged  : {plugin.brake_engaged}")
    print("  by model group :")
    for g, c in sorted(plugin.by_group.items(), key=lambda kv: -kv[1]):
        print(f"      {g:14s} ${c:.6f}")

await ledger_demo()

### 8.2 The deployment filter plugin

`async_filter_deployments` runs **inside** the Router, before strategy selection, and receives the list of healthy deployments. Whatever you return is the eligible set.

This is the escape hatch for eligibility rules that `enable_tag_filtering` cannot express: time of day, per tenant contracts, a kill switch, a compliance hold, or a budget brake that removes the expensive groups instead of failing the request.

**Torchlight's rule:** when the budget brake is engaged, do not fail. Remove `tl-guard` from eligibility for everything except safety traffic. Cost control should degrade the service, not break it.

In [ ]:
class EligibilityPlugin(CustomLogger):
    """Router level eligibility. Runs before load balancing, shrinks the candidate set."""

    def __init__(self, ledger: TorchlightPlugin):
        super().__init__()
        self.ledger = ledger
        self.blocked_regions: set[str] = set()
        self.decisions: list[str] = []

    async def async_filter_deployments(self, model, healthy_deployments, messages,
                                       request_kwargs=None, parent_otel_span=None):
        req = request_kwargs or {}
        is_safety = ((req.get("metadata") or {}).get("traffic_class") == "safety")
        kept = []
        for d in healthy_deployments:
            dep_id = d.get("model_info", {}).get("id", "?")
            region = d.get("litellm_params", {}).get("aws_region_name", "")

            if region in self.blocked_regions:
                self.decisions.append(f"drop {dep_id}: region {region} on compliance hold")
                continue
            if self.ledger.brake_engaged and model == "tl-guard" and not is_safety:
                self.decisions.append(f"drop {dep_id}: budget brake, tl-guard is safety-only")
                continue
            kept.append(d)

        # never return an empty set. An empty set is an outage, a full set is a bad month.
        return kept or healthy_deployments

elig = EligibilityPlugin(plugin)
litellm.callbacks = [plugin, elig]

rule("Eligibility plugin: compliance hold and budget brake")

async def eligibility_demo():
    r = build_router(num_retries=0)

    elig.blocked_regions = {"us-west-2"}
    seen = collections.Counter()
    for _ in range(12):
        res = await r.acompletion(model="tl-fast", messages=[{"role": "user", "content": "x"}], max_tokens=20)
        seen[res._hidden_params["model_id"]] += 1
    print("  compliance hold on us-west-2 ->", dict(seen))
    elig.blocked_regions = set()

    print(f"\n  budget brake engaged: {plugin.brake_engaged}")
    res = await r.acompletion(model="tl-guard", messages=[{"role": "user", "content": "routine question"}],
                              max_tokens=20)
    print("  tl-guard, routine traffic  ->", res._hidden_params["model_id"])
    res = await r.acompletion(model="tl-guard", messages=[{"role": "user", "content": "fire in the engine bay"}],
                              max_tokens=20, metadata={"traffic_class": "safety"})
    print("  tl-guard, safety traffic   ->", res._hidden_params["model_id"], "(safety exempt from the brake)")
    print("\n  filter decisions log:")
    for d in elig.decisions[-4:]:
        print("     .", d)
    print("\n  Read the routine-traffic line carefully. The brake DID drop tl-guard's only")
    print("  deployment, and `kept or healthy_deployments` handed it back. That is deliberate:")
    print("  a cost rule must degrade the service, never manufacture an outage. In production")
    print("  the correct move is to have somewhere cheaper to fall back TO, not to fail.")

await eligibility_demo()

### 8.3 Caching, and where it lies to you

`litellm.cache` gives you exact prompt caching in one line. On Torchlight's class A traffic (55%, highly repetitive: "where is my van") the hit rate is real money.

| Setting | Effect |
|---|---|
| `Cache(type="local")` | in process dict, per pod, dies on restart |
| `Cache(type="redis", host=..., port=...)` | shared across pods, survives restarts |
| `ttl=` | how long an entry lives |
| `caching=True` per call | opt in per request rather than globally |

**Three ways a cache lies to a roadside desk:**

1. **Staleness.** "Technician ETA 22 minutes" cached for 300 seconds is wrong 300 seconds later. Cache the *reasoning*, never the *live fact*.
2. **Cross tenant leakage.** The cache key is the prompt. If two fleet customers ask the same question, the second gets the first one's answer. Namespace by tenant or do not cache tool augmented answers at all.
3. **Cost dashboards go green for the wrong reason.** A cache hit reports near zero cost. Your spend drops, and you conclude the routing work paid off, when in fact one customer is refreshing the page.

**Torchlight's rule:** cache class C adjudications (contract text does not change) with a long TTL and a tenant namespace. Never cache class A (live ETAs). Never cache class D (safety, always fresh, always reasoned).

In [ ]:
rule("Caching: hit, miss, and the staleness trap")
from litellm.caching.caching import Cache

litellm.cache = Cache(type="local", ttl=300)

CLAUSE = [{"role": "user", "content": "Under FL-2291 clause 7.2, is a motorway tow covered?"}]
LIVE   = [{"role": "user", "content": "Where is technician EN-14 right now?"}]

a = litellm.completion(model=HAIKU, messages=CLAUSE, caching=True, **mock("Covered, GBP 50 excess."))
b = litellm.completion(model=HAIKU, messages=CLAUSE, caching=True, **mock("Covered, GBP 50 excess."))
print("  adjudication call 1 cache_hit:", a._hidden_params.get("cache_hit"))
print("  adjudication call 2 cache_hit:", b._hidden_params.get("cache_hit"), " <- correct, contracts do not move")
print("  same response id            :", a.id == b.id)

c = litellm.completion(model=MICRO, messages=LIVE, caching=False, **mock("EN-14 is on the A34, 22 min."))
d = litellm.completion(model=MICRO, messages=LIVE, caching=False, **mock("EN-14 has arrived."))
print("\n  live ETA call 1 cache_hit   :", c._hidden_params.get("cache_hit"))
print("  live ETA call 2 cache_hit   :", d._hidden_params.get("cache_hit"), " <- correct, caching=False on live facts")
print("  answers differ              :", c.choices[0].message.content != d.choices[0].message.content)

litellm.cache = None

### Section 8 recap

| Plugin | Hook | What it protects |
|---|---|---|
| PII redaction | `log_pre_api_call` | data leaving your boundary |
| Cost ledger | `async_log_success_event` | attribution per model group |
| Error counters | `async_log_failure_event` | which failure class is trending |
| Silent degradation detector | `log_success_fallback_event` | the outage your green dashboard is hiding |
| Eligibility filter | `async_filter_deployments` | residency, kill switches, budget brakes |

**Two rules to carry forward:**

1. `litellm.max_budget` does not enforce. Verified. Build the guard or use the proxy.
2. A filter plugin must never return an empty candidate set. Degrade the service, do not manufacture an outage.

---
# S9. The Terminal: Proxy and Gateway

Everything so far lived **inside your Python process**. That is the ceiling you eventually hit, and it is worth naming the moment precisely.

| Question | Router (in process) | Proxy (a service) |
|---|---|---|
| Which model serves this call? | yes | yes |
| Load balancing and fallbacks | yes | yes |
| Who spent what, across teams? | no, each process has its own counter | yes |
| Can a Node or Java service use it? | no | yes, it is an OpenAI compatible HTTP API |
| Can I rotate a model without a deploy? | no, config is in your code | yes, edit config, reload |
| Can I revoke one team's access at 02:00? | no | yes, revoke the key |
| Central audit log of every prompt | you build it per service | one place |

**Proxy and Gateway are the same process, different job descriptions.** People argue about this and it wastes time, so define it once:

- **Proxy** is the *mechanism*: an HTTP server that speaks the OpenAI API and forwards to providers.
- **Gateway** is the *role*: the single enforced entry point where policy lives (keys, budgets, guardrails, residency, audit).

A proxy that only one service uses is a proxy. The same proxy, once it is the mandatory path for every service and every team, is a gateway. **Nothing in the software changes. What changes is that bypassing it becomes a policy violation.**

---
### Challenge 9.A

Torchlight has three consumers: the mobile app (Python), the fleet portal (Node), and the ops console (internal, Java). Finance wants per team cost. Security wants prompts audited. Product wants to swap a model without a release.

| Option | Design |
|---|---|
| **A** | Each service embeds `litellm.Router` with the same `model_list`, shipped as a shared config file |
| **B** | Run the LiteLLM Proxy, all three services call it over HTTP with team scoped virtual keys |
| **C** | Build an internal REST wrapper around `litellm.Router` in Python, and have Node and Java call that |

### Verdict 9.A

**B**, and C is worth understanding because C is what teams build when they have not read the proxy docs.

- **A** fails on the first requirement. Three processes, three cost counters, no aggregation. It also fails "swap without a release": the shared config file still ships inside each service. And Node and Java cannot import a Python object at all, so A is not even possible here.
- **C** is exactly the LiteLLM Proxy, rebuilt by you, badly. You will reimplement virtual keys, budgets, rate limits, retries, health endpoints, spend logs and an OpenAI compatible surface. Every one of those is a month of work and a source of bugs. The tell that you are in this trap: you find yourself writing a `/v1/chat/completions` handler.
- **B** is the answer. One binary, one config file, an OpenAI compatible endpoint that every language already has a client for.

**The one honest cost of B:** you have added a network hop and a component that can fail. That is real. It is bought back by the fact that the hop is where you can now see, cap and control everything. Run it as two replicas behind a load balancer with a shared Redis and the availability argument goes away.

### 9.1 The full Torchlight gateway config

This is the whole system from S6, S7 and S8 expressed as configuration instead of code. Read it against the HLD in S0.4.

```mermaid
flowchart TB
    subgraph C[Callers]
      A1[mobile app - python]
      A2[fleet portal - node]
      A3[ops console - java]
    end
    A1 -->|sk-torch-consumer| PX
    A2 -->|sk-torch-fleet| PX
    A3 -->|sk-torch-ops| PX
    subgraph PX[LiteLLM Proxy :4000]
      K[virtual keys<br/>budgets, rpm, allowed models]
      R[router: strategy, fallbacks, cooldowns]
      CB[callbacks: your plugins from S8]
      CA[redis cache]
    end
    PX --> B1[Bedrock us-east-1]
    PX --> B2[Bedrock us-west-2]
    PX --> DB[(postgres<br/>spend logs, keys)]
    PX --> RD[(redis<br/>shared rpm/tpm, cache)]
```

In [ ]:
# ============================================================
# S9. Write the gateway config, then prove the model_list half of it actually loads.
# ============================================================
CONFIG_YAML = """
model_list:
  # ---------------- tl-fast : two regions, 3:1 canary weighting ----------------
  - model_name: tl-fast
    litellm_params:
      model: bedrock/us.amazon.nova-micro-v1:0
      aws_region_name: us-east-1
      weight: 3
      rpm: 600
      tags: ["general"]
    model_info:
      id: micro-use1
      max_input_tokens: 128000

  - model_name: tl-fast
    litellm_params:
      model: bedrock/us.amazon.nova-micro-v1:0
      aws_region_name: us-west-2
      weight: 1
      rpm: 200
      tags: ["general"]
    model_info:
      id: micro-usw2
      max_input_tokens: 128000

  # ---------------- tl-balanced : 300k context, vision ----------------
  - model_name: tl-balanced
    litellm_params:
      model: bedrock/us.amazon.nova-lite-v1:0
      aws_region_name: us-east-1
      rpm: 400
      tags: ["general", "long-context"]
    model_info:
      id: lite-use1
      max_input_tokens: 300000

  # ---------------- tl-deep : reasoning and tools ----------------
  - model_name: tl-deep
    litellm_params:
      model: bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0
      aws_region_name: us-east-1
      rpm: 200
      tags: ["general"]
    model_info:
      id: haiku-use1
      max_input_tokens: 200000

  # ---------------- tl-guard : safety critical only ----------------
  - model_name: tl-guard
    litellm_params:
      model: bedrock/us.anthropic.claude-sonnet-4-5-20250929-v1:0
      aws_region_name: us-east-1
      rpm: 60
      tags: ["safety"]
    model_info:
      id: sonnet-use1
      max_input_tokens: 200000

router_settings:
  routing_strategy: simple-shuffle
  num_retries: 2
  timeout: 30
  cooldown_time: 20
  allowed_fails: 2
  enable_pre_call_checks: true
  enable_tag_filtering: true
  redis_host: os.environ/REDIS_HOST      # shared rpm/tpm across replicas
  redis_port: os.environ/REDIS_PORT

  fallbacks:
    - tl-fast: ["tl-deep"]
    - tl-deep: ["tl-guard"]
  context_window_fallbacks:
    - tl-fast: ["tl-balanced"]
    - tl-deep: ["tl-balanced"]
  content_policy_fallbacks:
    - tl-fast: ["tl-guard"]

litellm_settings:
  drop_params: true
  set_verbose: false
  cache: true
  cache_params:
    type: redis
    ttl: 3600
    supported_call_types: ["acompletion", "completion"]
  callbacks: ["torchlight_plugins.instance"]   # your S8 plugin, loaded by dotted path
  failure_callback: ["langfuse"]

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY
  database_url: os.environ/DATABASE_URL       # spend logs, virtual keys, teams
  alerting: ["slack"]
  proxy_batch_write_at: 60
"""

with open("torchlight_config.yaml", "w") as f:
    f.write(CONFIG_YAML)
print("wrote torchlight_config.yaml")

In [ ]:
rule("Validating the config, not just eyeballing it")
import yaml

cfg = yaml.safe_load(CONFIG_YAML)
print("  YAML parses          : yes")
print("  model groups         :", sorted({m["model_name"] for m in cfg["model_list"]}))
print("  deployments          :", len(cfg["model_list"]))

# The model_list half is the piece that most often has a typo. Load it into a real Router.
probe_list = []
for m in cfg["model_list"]:
    lp = dict(m["litellm_params"])
    if MOCK:
        lp["mock_response"] = f"served by {m['model_info']['id']}"
    probe_list.append({"model_name": m["model_name"], "litellm_params": lp, "model_info": m["model_info"]})

rs = {k: v for k, v in cfg["router_settings"].items() if not str(v).startswith("os.environ/")}
probe = Router(model_list=probe_list, **rs)
print("  Router built from it : yes")

for g in sorted({m["model_name"] for m in cfg["model_list"]}):
    r = await probe.acompletion(model=g, messages=[{"role": "user", "content": "ping"}], max_tokens=20)
    print(f"    {g:12s} -> {r._hidden_params['model_id']}")
print("\n  Do this in CI. A typo in a model string is a 3am page, and it is catchable in 200ms.")

### 9.2 Running it, and the endpoints that matter

```bash
pip install "litellm[proxy]"

export LITELLM_MASTER_KEY="sk-torch-master-..."
export DATABASE_URL="postgresql://..."       # required for keys, teams and spend logs
export REDIS_HOST="..." REDIS_PORT="6379"
export AWS_REGION_NAME="us-east-1"

litellm --config torchlight_config.yaml --port 4000
```

| Endpoint | Use |
|---|---|
| `POST /v1/chat/completions` | the OpenAI compatible surface every client already speaks |
| `GET /health` | per deployment health, the one your load balancer should not use |
| `GET /health/liveliness` | cheap liveness for the load balancer |
| `GET /v1/models` | which groups a given key is allowed to see |
| `POST /key/generate` | mint a virtual key with a budget and model allow list |
| `GET /spend/logs` | per key, per model, per request spend |
| `/ui` | the admin console |

**Use `/health/liveliness` for your load balancer, not `/health`.** `/health` actually pings every deployment. Point a 1 second load balancer probe at it and you have built a self inflicted denial of service against your own Bedrock quota.

### 9.3 Virtual keys: where "gateway" stops being a word

This is the part that makes the whole architecture worth the network hop.

```bash
# Consumer app: cheap groups only, hard monthly cap, high throughput
curl -X POST http://localhost:4000/key/generate \
  -H "Authorization: Bearer $LITELLM_MASTER_KEY" \
  -H "Content-Type: application/json" \
  -d '{
    "key_alias": "torchlight-consumer-app",
    "team_id": "consumer",
    "models": ["tl-fast", "tl-balanced", "tl-guard"],
    "max_budget": 400,
    "budget_duration": "30d",
    "rpm_limit": 3000,
    "metadata": {"traffic_class": "consumer"}
  }'

# Fleet portal: needs the long context group, EU residency tag, lower rpm
curl -X POST http://localhost:4000/key/generate \
  -H "Authorization: Bearer $LITELLM_MASTER_KEY" \
  -d '{
    "key_alias": "torchlight-fleet-portal",
    "team_id": "fleet",
    "models": ["tl-balanced", "tl-deep"],
    "max_budget": 900,
    "budget_duration": "30d",
    "rpm_limit": 400,
    "tags": ["eu-resident"]
  }'
```

Read what those two blobs bought you, against the Challenge 9.A requirements:

| Requirement | Where it is now enforced |
|---|---|
| Per team cost | `team_id` plus `max_budget`, aggregated in Postgres |
| Model access control | `models` allow list. The consumer app **cannot** call `tl-deep` even with a bug |
| Data residency | `tags` on the key, matched against deployment tags by `enable_tag_filtering` |
| Revoke at 02:00 | `POST /key/delete`. No deploy, no restart |
| Rate limiting | `rpm_limit` per key, shared across replicas via Redis |

**The line that matters:** the consumer app cannot spend the fleet budget, cannot reach the expensive group, and cannot leave the EU, **regardless of what the application code does**. That is what "the gateway is where policy lives" means in practice.

### 9.4 Pointing anything at it

Once the proxy is up, every client in every language is one base URL change away.

| Client | Change |
|---|---|
| OpenAI Python SDK | `OpenAI(base_url="http://proxy:4000/v1", api_key="sk-torch-consumer")` |
| LiteLLM SDK | `model="litellm_proxy/tl-fast"`, `api_base="http://proxy:4000"` |
| Strands (S10) | `LiteLLMModel(model_id="litellm_proxy/tl-fast", client_args={...})` |
| LangChain (S11) | `ChatLiteLLM(model="litellm_proxy/tl-fast", api_base=..., api_key=...)` |
| Node, Java, Go | any OpenAI client, point it at the base URL |

**This is the payoff of the whole notebook.** Your Strands agent and your LangGraph desk stop knowing anything about Bedrock, regions, weights, fallbacks or budgets. They ask for `tl-fast`. The gateway is the only thing that knows what that means, and it can change its mind without touching them.

In [ ]:
rule("The same call, three ways to reach it")

print("""  1) direct SDK, no indirection
       litellm.completion(model="bedrock/us.amazon.nova-micro-v1:0", ...)
       app knows: provider, region, model version, and nothing about failure

  2) in-process Router
       router.acompletion(model="tl-fast", ...)
       app knows: the job name. Router knows the rest. Cost is per process.

  3) through the gateway
       litellm.completion(model="litellm_proxy/tl-fast",
                          api_base="http://proxy:4000", api_key="sk-torch-consumer")
       app knows: the job name and its own key. Everything else is policy,
       enforced for every language and every team, changeable without a deploy.

  Each step removes knowledge from the application. That is the whole point.""")

### Section 9 recap

| Decision | Answer |
|---|---|
| One service, one team, Python only | in process `Router` is enough |
| Second language, second team, or a finance question | Proxy |
| Proxy vs gateway | same process, different mandate. Gateway means bypassing it is a violation |
| Load balancer probe | `/health/liveliness`, never `/health` |
| Config safety | build a `Router` from the `model_list` in CI, catch typos in 200ms |
| Enforcement point | virtual keys: budget, model allow list, rpm, residency tags |

---
# S10. Strands on LiteLLM

You already know Strands. What changes when its model provider is LiteLLM rather than `BedrockModel` is worth being precise about, because one of the changes is silent and will break your error handling.

### 10.1 Two ways to wire it, and when to use each

```mermaid
flowchart TB
    subgraph A["Pattern A: in-process"]
      AG1[Strands Agent] --> LM1["LiteLLMModel(model_id='bedrock/us...')"]
      LM1 --> BR1[Bedrock]
    end
    subgraph B["Pattern B: gateway-backed"]
      AG2[Strands Agent] --> LM2["LiteLLMModel(model_id='litellm_proxy/tl-fast',<br/>client_args={api_base, api_key})"]
      LM2 --> PX[LiteLLM Proxy]
      PX --> RT[Router: LB, fallbacks, budgets, keys]
      RT --> BR2[Bedrock x N]
    end
```

| | Pattern A, in process | Pattern B, gateway backed |
|---|---|---|
| Setup | one line | a running proxy |
| Load balancing and fallbacks | you get none unless you build them | all of S6 for free |
| Cost attribution | per process | per key, per team |
| Model swap | code change | config change |
| Use when | notebook, prototype, single agent script | anything a customer touches |

**Pattern B is the answer for Torchlight**, and note what it does to the agent: the agent asks for `tl-fast` and knows nothing else. Every decision from S6 to S9 applies to it without the agent importing a single line of that logic.

In [ ]:
# ============================================================
# S10. Pattern A, running fully offline.
# ============================================================
from strands import Agent, tool
from strands.models.litellm import LiteLLMModel
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.types.exceptions import (ContextWindowOverflowException,
                                      ModelThrottledException, EventLoopException)

def tl_model(model_id: str, canned: str | None = None, **params):
    p = {"max_tokens": 400, "temperature": 0.2, **params}
    if MOCK and canned is not None:
        p["mock_response"] = canned
    return LiteLLMModel(model_id=model_id, params=p)

# Pattern B, for reference. Uncomment when a proxy is actually running.
# gateway_model = LiteLLMModel(
#     model_id="litellm_proxy/tl-fast",
#     client_args={"api_base": "http://localhost:4000", "api_key": os.environ["TORCHLIGHT_KEY"]},
#     params={"max_tokens": 400},
# )

print("LiteLLMModel wired. Pattern A active, Pattern B is a two line change.")

### 10.2 Tools, and the exception rule everyone gets wrong

Torchlight's dispatch agent needs tools that hit real systems: the case store, the technician tracker, the contract index. Real systems fail.

**Strands contains a raised exception automatically** and hands the model a `status="error"` tool result. That is good default behaviour and it is verified below. It is also **not enough**, and here is the distinction that matters:

| Failure kind | Correct tool behaviour | Why |
|---|---|---|
| **Expected domain outcome** ("no such case") | `return` a structured result the model can reason about | the model should say "I cannot find that case, check the reference" |
| **Transient infrastructure** (timeout, 503) | `raise`, let Strands contain it, and let the model retry or hand off | the model should not invent an answer |
| **Data integrity** (contract index returned corrupt rows) | `raise` **and** set a flag your application checks | the model must never present corrupt data as fact |

**The anti pattern:** `except Exception: return "no data found"`. That collapses all three into the same signal. The model then cheerfully reports "no matching contract" when the truth was "the contract database is down", and Torchlight declines a valid claim.

In [ ]:
# ============================================================
# S10. Tools that fail honestly.
# ============================================================
CASES = {"TL-88421": {"vehicle": "AB12 CDE", "fault": "no start", "tech": "EN-14", "eta_min": 22}}
OUTAGE = {"contract_index": False}      # flip to simulate an outage

@tool
def lookup_case(case_id: str) -> dict:
    """Look up a Torchlight roadside case by its reference, for example TL-88421."""
    if case_id in CASES:
        return {"status": "success", "content": [{"text": json.dumps(CASES[case_id])}]}
    # EXPECTED domain outcome. Not an exception. The model should reason about it.
    return {"status": "success",
            "content": [{"text": json.dumps({"found": False, "hint": "reference format is TL-#####"})}]}

@tool
def technician_eta(case_id: str) -> str:
    """Live ETA in minutes for the technician assigned to a case."""
    if case_id not in CASES:
        return json.dumps({"found": False})
    raise TimeoutError("tracker service did not respond in 2s")     # TRANSIENT. Raise.

@tool
def contract_clause(contract_id: str, topic: str) -> str:
    """Retrieve the clause of a fleet contract covering a given topic."""
    if not OUTAGE["contract_index"]:
        # DATA INTEGRITY / availability. Raise AND make it unmistakable.
        raise RuntimeError("CONTRACT_INDEX_UNAVAILABLE: refuse to answer coverage from memory")
    return json.dumps({"contract": contract_id, "clause": "7.2", "covered": True, "excess_gbp": 50})

rule("Three tools, three different failure shapes")

agent = Agent(model=tl_model(MICRO, canned="ack"),
              tools=[lookup_case, technician_eta, contract_clause],
              system_prompt="You are the Torchlight dispatch desk.",
              callback_handler=None)

print("  expected-miss  :", json.dumps(agent.tool.lookup_case(case_id="TL-00000"))[:130])
print("  transient      :", json.dumps(agent.tool.technician_eta(case_id="TL-88421"))[:130])
print("  integrity      :", json.dumps(agent.tool.contract_clause(contract_id="FL-2291", topic="tow"))[:130])
print("\n  All three came back as tool results, not crashes. Note that the LAST TWO carry")
print("  status='error' while the first is a successful 'not found'. The model can tell")
print("  them apart. With `except: return 'no data'` it could not.")

---
### Challenge 10.A

Your Strands dispatch agent has been running on `BedrockModel` with this handler:

```python
try:
    result = agent(user_message)
except ModelThrottledException:
    return "We are busy, please hold."
```

You migrate the provider to `LiteLLMModel` to get the Router. Nothing else changes. What breaks?

| Option | Prediction |
|---|---|
| **A** | Nothing, LiteLLM raises the same Strands exceptions |
| **B** | Throttling now escapes as a raw `litellm.RateLimitError`, the handler misses it, and the request 500s |
| **C** | Throttling is now silently swallowed and the user gets an empty answer |

### Verdict 10.A

**B**, and it is verified in the cell below.

`BedrockModel` translates Bedrock's `ThrottlingException` into Strands' own `ModelThrottledException`. **`LiteLLMModel` does not.** A LiteLLM `RateLimitError` propagates out of `agent()` unchanged. Your `except ModelThrottledException` block becomes dead code on the day you migrate, and you find out during the first traffic spike.

**Two things to take from this, and the second one is the important one.**

1. Narrowly: after migrating to `LiteLLMModel`, catch `litellm.RateLimitError` too. Or catch both.
2. Broadly, and this is the architectural point: **you should not be handling throttling in the agent at all.** Throttling is a transport concern. If the agent is seeing `RateLimitError`, your Router is missing retries, a second region, or a fallback group. Fix it one layer down and the agent never learns the word "throttle".

Interestingly, context overflow **is** translated: LiteLLM's `ContextWindowExceededError` arrives as Strands' `ContextWindowOverflowException`, which is what triggers the conversation manager to reduce context. So the mapping is partial, not absent, which is exactly the kind of thing you only discover by testing it.

In [ ]:
rule("Verified: which failures Strands translates and which leak through")

def strands_failure(inject: str):
    a = Agent(model=tl_model(MICRO, canned=inject), callback_handler=None,
              conversation_manager=SlidingWindowConversationManager(window_size=4))
    try:
        a("status of TL-88421?")
        return "no raise"
    except Exception as e:
        return type(e).__name__

r1 = strands_failure("litellm.ContextWindowExceededError")
r2 = strands_failure("litellm.RateLimitError")
r3 = strands_failure("litellm.InternalServerError")

print(f"  litellm.ContextWindowExceededError -> {r1:32s} TRANSLATED to a Strands exception")
print(f"  litellm.RateLimitError             -> {r2:32s} LEAKS THROUGH as a LiteLLM exception")
print(f"  litellm.InternalServerError        -> {r3:32s} LEAKS THROUGH")
print()
print("  So this handler is a bug after migrating:")
print("      except ModelThrottledException: ...")
print("  And this one is correct:")
print("      except (ModelThrottledException, litellm.RateLimitError): ...")
print()
print("  And the correct FIX is neither: put retries and a fallback group in the Router,")
print("  so the agent never sees a throttle at all.")

### 10.3 Tier escalation inside a live conversation

The cascade from S7.6, applied to an agent. The clean in process mechanism is that **`agent.model` is swappable at runtime and the message history survives the swap** (verified below).

That gives you a pattern the frameworks do not advertise:

```mermaid
flowchart LR
    T1["turn 1..n on tl-fast<br/>Nova Micro"] --> V{verifier or<br/>user frustration signal}
    V -- ok --> T1
    V -- escalate --> SWAP["agent.model = tl-deep model<br/>history preserved"]
    SWAP --> T2["turn n+1 on tl-deep<br/>Haiku 4.5, full context"]
```

The strong model inherits everything that already happened, including failed tool calls, which is exactly the context it needs to do better than the cheap model did.

In [ ]:
rule("Escalation mid-conversation, with history preserved")

desk = Agent(
    model=tl_model(MICRO, canned="Case TL-88421 is open. I am not sure what the fault is."),
    tools=[lookup_case, technician_eta, contract_clause],
    system_prompt="You are the Torchlight dispatch desk. Be terse and factual.",
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    callback_handler=None,
)

r1 = desk("What is happening with TL-88421?")
print("  tier 1 (tl-fast)  :", str(r1).strip()[:76])
print("  messages so far   :", len(desk.messages))

ok, why = structural_verify(str(r1))
print(f"  verifier          : {'PASS' if ok else 'ESCALATE'} ({why})")

if not ok:
    desk.model = tl_model(HAIKU, canned=(
        "Case TL-88421: no-start fault on AB12 CDE. Symptoms match alternator failure. "
        "Technician EN-14 dispatched with a jump pack, ETA 22 minutes."))
    r2 = desk("Try again with the full case context.")
    print("  tier 2 (tl-deep)  :", str(r2).strip()[:76])

print("  messages after    :", len(desk.messages), "(history survived the swap)")
print("  model now         :", desk.model.get_config()["model_id"].split('/')[-1])
print("\n  The strong model saw the weak model's failed attempt. That is the whole value:")
print("  escalation without re-asking the customer to repeat themselves.")

### 10.4 When memory fails

"Memory" in a Strands agent is three separate things that fail three separate ways. Conflating them is why "our agent forgot everything" incidents take so long to diagnose.

| Layer | What it is | Failure | Correct response |
|---|---|---|---|
| **Working memory** | `agent.messages`, the live list | grows past the context window | conversation manager trims or summarises |
| **Conversation manager** | `SlidingWindowConversationManager`, `SummarizingConversationManager` | the summarising one makes a **model call**, which can fail | fall back to sliding window, never fail the turn |
| **Session persistence** | `FileSessionManager`, `S3SessionManager` | store unreachable, or concurrent writes | serve the turn stateless, flag the degradation |

```mermaid
flowchart TB
    T([turn arrives]) --> LOAD{load session}
    LOAD -- ok --> CTX{context within window?}
    LOAD -- store down --> DEG["degrade: stateless turn<br/>tell the user, do not pretend"]
    DEG --> CTX
    CTX -- yes --> RUN[run the agent]
    CTX -- no --> RED{reduce context}
    RED -- summariser ok --> RUN
    RED -- summariser failed --> SLIDE["fall back to sliding window<br/>drop oldest turns"]
    SLIDE --> RUN
    RUN --> SAVE{persist session}
    SAVE -- ok --> DONE([respond])
    SAVE -- store down --> WARN["respond, but mark the turn unpersisted<br/>next turn starts cold"]
    WARN --> DONE
```

**Two rules that come out of that diagram:**

1. **A memory failure must never fail the turn.** A roadside caller in distress does not care that your session store is down. Serve them statelessly and say so.
2. **A summarising conversation manager is a model call, and therefore has every failure mode in S5.** If it is not wrapped, your memory layer inherits your provider's throttling. Give it the cheapest model group and a fallback.

In [ ]:
# ============================================================
# S10. Memory that degrades instead of failing.
# ============================================================
class FlakySessionStore:
    """Stands in for FileSessionManager / S3SessionManager."""
    def __init__(self): self.data, self.up = {}, True
    def load(self, sid):
        if not self.up: raise ConnectionError("session store unreachable")
        return self.data.get(sid, [])
    def save(self, sid, msgs):
        if not self.up: raise ConnectionError("session store unreachable")
        self.data[sid] = list(msgs)

store = FlakySessionStore()

@dataclass
class TurnResult:
    text: str
    degraded: list[str] = field(default_factory=list)

def resilient_turn(session_id: str, user_text: str, model, canned=None) -> TurnResult:
    degraded, history = [], []

    # 1. load, degrade to stateless on failure
    try:
        history = store.load(session_id)
    except Exception as e:
        degraded.append(f"session load failed ({type(e).__name__}), running stateless")

    a = Agent(model=model, tools=[lookup_case], messages=history,
              conversation_manager=SlidingWindowConversationManager(window_size=20),
              system_prompt="You are the Torchlight dispatch desk.", callback_handler=None)

    # 2. run, and treat context overflow as a memory event rather than a model event
    try:
        out = str(a(user_text))
    except ContextWindowOverflowException:
        degraded.append("context overflow, trimming to the last 4 turns and retrying")
        a.messages = a.messages[-4:]
        out = str(a(user_text))

    # 3. persist, and be explicit when we could not
    try:
        store.save(session_id, a.messages)
    except Exception as e:
        degraded.append(f"session save failed ({type(e).__name__}), next turn starts cold")

    return TurnResult(out, degraded)

rule("Memory degradation, three failure modes, zero failed turns")

m_ok = tl_model(MICRO, canned="Case TL-88421: technician EN-14, ETA 22 minutes.")

t = resilient_turn("s-1", "status of TL-88421?", m_ok)
print("  1) healthy store  :", t.text.strip()[:58], "| degraded:", t.degraded)

store.up = False
t = resilient_turn("s-1", "and the ETA?", m_ok)
print("  2) store down     :", t.text.strip()[:58])
for d in t.degraded: print("       .", d)
store.up = True

t = resilient_turn("s-1", "recap everything", m_ok)
print("  3) store back     :", t.text.strip()[:58], "| degraded:", t.degraded)
print("\n  Three turns, one outage, zero exceptions reached the caller.")
print("  And note turn 2 is HONEST about being stateless rather than silently forgetting.")

In [ ]:
rule("The summarising conversation manager is a model call, so it can fail")

print("""  SummarizingConversationManager(summarization_agent=Agent(model=<a model>))
  makes a real LLM call every time it compacts. That means it inherits:
    RateLimitError, Timeout, ContextWindowExceededError, ContentPolicyViolationError

  Torchlight's configuration:
    summarisation agent  -> tl-fast through the Router, with fallbacks
                            (never the same group as the answering agent, or a
                             capacity event takes out memory AND answers together)
    on summariser failure -> fall back to SlidingWindowConversationManager
                            losing old turns beats failing the turn

  The failure nobody plans for: the summariser summarises a SAFETY exchange into
  'customer reported a vehicle issue' and the escalation context is gone. Pin the
  safety-relevant turns. SlidingWindowConversationManager has `pin_first` for
  exactly this, and Summarizing has `preserve_recent_messages`.""")

### 10.5 Hooks: Strands observability that pairs with the S8 plugins

Strands has its own hook system, and it sees things LiteLLM callbacks cannot: tool selection, message construction, the agent loop. Use both. They answer different questions.

| Question | Answered by |
|---|---|
| Which deployment served this, what did it cost, did a fallback rescue it | LiteLLM callbacks (S8) |
| Which tool did the agent pick, did it fail, how many loop iterations | Strands hooks |

In [ ]:
from strands.hooks import (HookProvider, HookRegistry, BeforeToolCallEvent,
                           AfterToolCallEvent, BeforeModelCallEvent)

class DispatchTrace(HookProvider):
    """Strands-side trace. Pairs with the LiteLLM plugin, does not duplicate it."""
    def __init__(self):
        self.events = []

    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(BeforeModelCallEvent, self._model)
        registry.add_callback(BeforeToolCallEvent, self._tool_start)
        registry.add_callback(AfterToolCallEvent, self._tool_end)

    def _model(self, event):
        self.events.append(("model_call", "-"))

    def _tool_start(self, event):
        self.events.append(("tool_start", (event.tool_use or {}).get("name", "?")))

    def _tool_end(self, event):
        res = event.result or {}
        exc = getattr(event, "exception", None)
        self.events.append(("tool_end", f"{res.get('status', '?')}"
                                        f"{' / ' + type(exc).__name__ if exc else ''}"))

rule("Strands hooks alongside the LiteLLM plugin")

trace = DispatchTrace()
a = Agent(model=tl_model(MICRO, canned="Technician EN-14, ETA 22 minutes."),
          tools=[lookup_case, technician_eta], hooks=[trace], callback_handler=None)

a("status of TL-88421?")                       # a model turn, no tool selected (mocked answer)
a.tool.lookup_case(case_id="TL-88421")         # a tool that succeeds
a.tool.technician_eta(case_id="TL-88421")      # a tool that raises

print("  strands-side events:")
for kind, detail in trace.events:
    print(f"      {kind:12s} {detail}")
print("\n  litellm-side truth :", {"errors": plugin.errors,
                                  "degradations": len(plugin.silent_degradations)})
print()
print("  AfterToolCallEvent also carries `exception` and a mutable `retry` flag, so a hook")
print("  can retry a transient tool failure WITHOUT the model ever seeing the error.")
print("  That is the tool-layer twin of what the Router does at the model layer.")
print("\n  Two traces, two layers, no overlap. Correlate them on a request id in production.")

### Section 10 recap

| Point | Detail |
|---|---|
| Two wiring patterns | in process for prototypes, `litellm_proxy/<group>` for anything real |
| Tool failures have three shapes | expected miss, transient, integrity. Never collapse them |
| `LiteLLMModel` does **not** translate throttling | `ModelThrottledException` handlers go dead on migration. Verified |
| Context overflow **is** translated | it reaches the conversation manager as designed |
| Escalate by swapping `agent.model` | history survives, the strong model sees the weak one's failure |
| Memory has three layers | working, manager, session. Each degrades, none fails the turn |
| The summariser is a model call | give it its own cheap group and its own fallback |

---
# S11. LangChain and LangGraph on LiteLLM

### 11.1 The import that every tutorial online gets wrong

> **Verified against the installed packages, and this will save you an afternoon.**
>
> `from langchain_community.chat_models import ChatLiteLLM` **no longer works.** `langchain-community` is being sunset and the class has been removed. In `langchain-community 0.4.2` the import raises `AttributeError`.
>
> The current package is **`langchain-litellm`**:
>
> ```python
> from langchain_litellm import ChatLiteLLM, ChatLiteLLMRouter
> ```
>
> Install with `pip install langchain-litellm`. Almost every blog post and answer you will find still shows the old path.

| | |
|---|---|
| `ChatLiteLLM` | a chat model bound to **one** LiteLLM model string |
| `ChatLiteLLMRouter` | a chat model bound to a **`litellm.Router`**, so it inherits load balancing, cooldowns and all three fallback ladders |
| `LiteLLMEmbeddings` | the embeddings equivalent |

**Use `ChatLiteLLMRouter` for anything real.** `ChatLiteLLM` gives you provider portability. `ChatLiteLLMRouter` gives you everything from S6 as well, and it is the same amount of code.

In [ ]:
# ============================================================
# S11. LangChain wiring. Verified against langchain-litellm 0.7.0.
# ============================================================
from langchain_litellm import ChatLiteLLM, ChatLiteLLMRouter
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool as lc_tool

rule("ChatLiteLLM vs ChatLiteLLMRouter")

# Pass-through kwargs go in model_kwargs, NOT as top level kwargs. That is the second trap.
plain = ChatLiteLLM(model=MICRO, **({"model_kwargs": {"mock_response": "[plain] EN-14, 22 min."}} if MOCK else {}))
print("  ChatLiteLLM        :", plain.invoke([HumanMessage("ETA for TL-88421?")]).content)

routed = ChatLiteLLMRouter(router=router, model_name="tl-fast")
print("  ChatLiteLLMRouter  :", routed.invoke([HumanMessage("ETA for TL-88421?")]).content)

In [ ]:
rule("The reason to prefer the Router binding: fallbacks come with it")

# tl-fast is down in both regions. The LangChain object does not know or care.
broken = build_router(fail={"micro-use1": "litellm.InternalServerError",
                            "micro-usw2": "litellm.InternalServerError"}, num_retries=0)
lc = ChatLiteLLMRouter(router=broken, model_name="tl-fast")
out = lc.invoke([HumanMessage("Diagnose a grinding noise from the front wheel.")])
print("  asked for tl-fast, both deployments 500ing")
print("  answer still arrived:", out.content[:66])
print("\n  Zero LangChain-side retry code. The fallback ladder from S6 did it.")
print("  Compare with `llm.with_retry()`, which retries the SAME model and cannot re-route.")

---
### Challenge 11.A

Torchlight's LangGraph desk needs resilience. Three proposals:

| Option | Design |
|---|---|
| **A** | `ChatLiteLLM(...).with_retry(stop_after_attempt=3)` on every node |
| **B** | `ChatLiteLLMRouter(router=..., model_name=...)` with the Router's fallbacks, plus LangGraph `RetryPolicy` on nodes |
| **C** | Both A and B, belt and braces |

### Verdict 11.A

**B.** And C is actively worse than B, which is the interesting part.

- **A** retries the same model string. It cannot re-route to another region, cannot promote on a context overflow, and cannot fall back on a refusal. It solves the narrowest slice of the failure taxonomy from S5.
- **C** stacks retries. If LangChain retries 3 times and the Router retries 2 times with a 2 hop fallback ladder, a single failing request can produce **18 provider calls**. That is a real incident pattern: your retry storm is indistinguishable from an attack, and you burn your quota fighting yourself. **Retry budgets multiply, they do not add.**
- **B** puts each concern in exactly one layer:

| Concern | Layer |
|---|---|
| provider failed, try another deployment or group | LiteLLM Router |
| the node's own code failed (a tool, a parse, a database) | LangGraph `RetryPolicy` |
| the whole graph is wedged | `recursion_limit` and a timeout above the graph |

**The rule:** *one retry budget per failure domain, and never two budgets on the same domain.* Before you add a retry anywhere, ask which layer already owns that failure.

### 11.2 LLD: the Torchlight desk as a graph

The decision layer from S7 becomes graph structure, which buys you something the Strands version does not have: **the route is inspectable state**, and you can time travel to a specific routing decision when a case is disputed.

```mermaid
flowchart TB
    START([START]) --> GUARD[guard_rules<br/>L1 heuristic, hard safety]
    GUARD --> SHAPE[shape_check<br/>L2 context and capability]
    SHAPE --> DECIDE{route}
    DECIDE -- safety --> GD[answer_guard<br/>tl-guard]
    DECIDE -- status --> FS[answer_fast<br/>tl-fast]
    DECIDE -- triage --> DP[answer_deep<br/>tl-deep + tools]
    DECIDE -- coverage --> BL[answer_balanced<br/>tl-balanced 300k]
    FS --> V[verify]
    DP --> V
    BL --> V
    GD --> RESP
    V -- pass --> RESP([respond])
    V -- fail and budget left --> ESC[escalate one tier]
    ESC --> DECIDE
    V -- fail and no budget --> HUM[human handoff]
    HUM --> RESP
```

Note `GD` bypasses `verify` and goes straight to respond. Safety answers are not subject to a cost driven escalation loop, and adding a verifier hop to them only adds latency to the traffic class with the tightest latency budget.

In [ ]:
# ============================================================
# S11. The Torchlight desk as a LangGraph state machine.
# ============================================================
import operator
from typing import Annotated, TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import RetryPolicy

class DeskState(TypedDict):
    messages: Annotated[list, add_messages]
    route: str
    # `operator.add` is the reducer. WITHOUT it every node REPLACES the trail and you
    # end up with only the last entry. This is the most common LangGraph state bug.
    trail: Annotated[list, operator.add]
    escalations: int
    degraded: Annotated[list, operator.add]

MAX_ESCALATIONS = 2

def llm_for(group: str, canned: str | None = None):
    m = ChatLiteLLMRouter(router=router, model_name=group)
    return m, ({"mock_response": canned} if (MOCK and canned) else {})

def guard_rules(state: DeskState) -> dict:
    text = state["messages"][-1].content
    d = l1_heuristic(text if isinstance(text, str) else "")
    if d and d.hard:
        return {"route": d.group, "trail": [f"L1 HARD -> {d.group}: {d.reason}"]}
    return {"route": d.group if d else "tl-deep",
            "trail": [f"L1 {'soft' if d else 'no match, safe-up default'} -> {d.group if d else 'tl-deep'}"]}

def shape_check(state: DeskState) -> dict:
    msgs = [{"role": "user", "content": m.content} for m in state["messages"]
            if isinstance(m.content, str)]
    c = l2_context(msgs, state["route"])
    if c.group != state["route"]:
        return {"route": c.group, "trail": [f"L2 CONSTRAINT -> {c.group}: {c.reason}"]}
    return {"trail": [f"L2 ok, {state['route']} satisfies constraints"]}

def pick(state: DeskState) -> Literal["answer_fast", "answer_balanced", "answer_deep", "answer_guard"]:
    return {"tl-fast": "answer_fast", "tl-balanced": "answer_balanced",
            "tl-deep": "answer_deep", "tl-guard": "answer_guard"}[state["route"]]

def _answer(state: DeskState, group: str, canned: str) -> dict:
    m, kw = llm_for(group, canned)
    out = m.invoke(state["messages"], **kw)
    return {"messages": [out], "trail": [f"answered on {group}"]}

def answer_fast(s):     return _answer(s, "tl-fast",     "Case TL-88421: EN-14 assigned, ETA 22 minutes.")
def answer_balanced(s): return _answer(s, "tl-balanced", "Case TL-88421: FL-2291 clause 7.2 covers the tow, GBP 50 excess.")
def answer_deep(s):     return _answer(s, "tl-deep",     "Case TL-88421: alternator suspected. EN-14 dispatched with a jump pack.")
def answer_guard(s):    return _answer(s, "tl-guard",    "Case TL-88421: leave the vehicle, stand behind the barrier. Police notified, EN-14 en route.")

def verify(state: DeskState) -> dict:
    ok, why = structural_verify(state["messages"][-1].content)
    return {"trail": [f"verify {'PASS' if ok else 'FAIL'}: {why}"]}

def after_verify(state: DeskState) -> Literal["escalate", "respond"]:
    ok, _ = structural_verify(state["messages"][-1].content)
    if ok:
        return "respond"
    return "escalate" if state["escalations"] < MAX_ESCALATIONS else "respond"

def escalate(state: DeskState) -> dict:
    nxt = promote(state["route"])
    return {"route": nxt, "escalations": state["escalations"] + 1,
            "trail": [f"ESCALATE {state['route']} -> {nxt}"]}

def respond(state: DeskState) -> dict:
    return {"trail": ["responded"]}

print("nodes defined")

In [ ]:
# ============================================================
# S11. A checkpointer that degrades instead of failing the turn.
# ============================================================
class ResilientSaver(InMemorySaver):
    """
    Wraps any checkpointer. A memory outage must not fail the turn (S10.4, rule 1).
    In production the parent would be PostgresSaver or a Redis-backed saver.
    """
    def __init__(self, healthy: bool = True):
        super().__init__()
        self.healthy = healthy
        self.degraded: list[str] = []

    def _down(self, op):
        self.degraded.append(op)
        return None

    def put(self, config, checkpoint, metadata, new_versions):
        if not self.healthy:
            self._down("put"); return config
        return super().put(config, checkpoint, metadata, new_versions)

    def put_writes(self, config, writes, task_id, task_path=""):
        if not self.healthy:
            return self._down("put_writes")
        return super().put_writes(config, writes, task_id, task_path)

    def get_tuple(self, config):
        if not self.healthy:
            return self._down("get_tuple")
        return super().get_tuple(config)

def build_desk(saver):
    g = StateGraph(DeskState)
    g.add_node("guard_rules", guard_rules)
    g.add_node("shape_check", shape_check)
    # RetryPolicy covers NODE failures (a tool, a parse). Provider failures are the Router's job.
    g.add_node("answer_fast",     answer_fast,     retry_policy=RetryPolicy(max_attempts=2))
    g.add_node("answer_balanced", answer_balanced, retry_policy=RetryPolicy(max_attempts=2))
    g.add_node("answer_deep",     answer_deep,     retry_policy=RetryPolicy(max_attempts=2))
    g.add_node("answer_guard",    answer_guard,    retry_policy=RetryPolicy(max_attempts=2))
    g.add_node("verify", verify)
    g.add_node("escalate", escalate)
    g.add_node("respond", respond)

    g.add_edge(START, "guard_rules")
    g.add_edge("guard_rules", "shape_check")
    g.add_conditional_edges("shape_check", pick,
        {"answer_fast": "answer_fast", "answer_balanced": "answer_balanced",
         "answer_deep": "answer_deep", "answer_guard": "answer_guard"})
    for n in ["answer_fast", "answer_balanced", "answer_deep"]:
        g.add_edge(n, "verify")
    g.add_edge("answer_guard", "respond")          # safety bypasses the cost-driven loop
    g.add_conditional_edges("verify", after_verify, {"escalate": "escalate", "respond": "respond"})
    g.add_conditional_edges("escalate", pick,
        {"answer_fast": "answer_fast", "answer_balanced": "answer_balanced",
         "answer_deep": "answer_deep", "answer_guard": "answer_guard"})
    g.add_edge("respond", END)
    return g.compile(checkpointer=saver)

saver = ResilientSaver(healthy=True)
desk_graph = build_desk(saver)
print("graph compiled")

In [ ]:
rule("The desk graph on three traffic classes")

async def run_desk(text, thread):
    out = await desk_graph.ainvoke(
        {"messages": [HumanMessage(text)], "route": "", "trail": [], "escalations": 0, "degraded": []},
        config={"configurable": {"thread_id": thread}, "recursion_limit": 20})
    print(f"\n  IN : {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"  ROUTE: {out['route']}   escalations: {out['escalations']}")
    print(f"  OUT: {out['messages'][-1].content[:72]}")
    for t in out["trail"]:
        print(f"       . {t}")
    return out

await run_desk("Where is my technician for TL-88421?", "t-a")
await run_desk("Broken down on the M1 hard shoulder with two kids in the car", "t-b")
# 220k tokens: fits tl-balanced (300k), does NOT fit tl-deep (200k) or tl-fast (128k)
await run_desk("Adjudicate TL-88421 against FL-2291. " + "clause " * 220_000, "t-c")

In [ ]:
rule("Memory outage: the graph degrades, the turn still completes")

sick = ResilientSaver(healthy=False)
sick_desk = build_desk(sick)
out = await sick_desk.ainvoke(
    {"messages": [HumanMessage("Where is my technician for TL-88421?")],
     "route": "", "trail": [], "escalations": 0, "degraded": []},
    config={"configurable": {"thread_id": "t-down"}, "recursion_limit": 20})

print("  turn completed  :", out["messages"][-1].content[:60])
print("  degraded ops    :", collections.Counter(sick.degraded))
print("\n  Without the wrapper this raises out of ainvoke and the caller gets a 500.")
print("  Compare against a raw InMemorySaver whose put() throws: verified in S10.4 terms,")
print("  the exception propagates and the turn dies.")
print("\n  What you MUST do alongside this: surface the degradation. A silently stateless")
print("  desk that keeps saying 'what is your case reference?' is its own incident.")

In [ ]:
rule("Time travel: why the graph version wins for disputed cases")

state = desk_graph.get_state({"configurable": {"thread_id": "t-b"}})
print("  final route    :", state.values["route"])
print("  full decision trail, replayable months later:")
for t in state.values["trail"]:
    print("      .", t)

hist = list(desk_graph.get_state_history({"configurable": {"thread_id": "t-b"}}))
print(f"\n  checkpoints kept: {len(hist)}")
print("  A fleet customer disputes an adjudication. You can replay the exact routing")
print("  decision, the exact model group, and the exact verification outcome.")
print("  The Strands version (S10) answers faster. This version is auditable.")

### 11.3 Choosing between the Strands desk and the LangGraph desk

Both were built on the same Router, the same fallbacks and the same decision layer. They differ in what they make easy.

| | Strands (S10) | LangGraph (S11) |
|---|---|---|
| Control flow | the model decides the loop | you decide the graph |
| Best at | open ended tool use, exploration | invariants, approval gates, ordering |
| Routing decision | in code around the agent | a node, in inspectable state |
| Audit | hooks and traces | checkpoints and time travel |
| Cost of a new branch | a new tool | a new node and edges |
| Torchlight fit | class B fault triage, where the next tool is unpredictable | class C adjudication, where the sequence is regulated |

**Torchlight runs both**, and this is the sentence worth writing on the whiteboard: **the graph owns business invariants, approval gates, sequencing and side effects; the agent owns bounded exploration where the next step cannot be predicted upfront.**

And the reason this notebook is about LiteLLM rather than about either framework: **both of them sit on the same Router, the same fallbacks, the same budgets and the same gateway.** The framework choice becomes reversible, which is the only kind of framework choice worth making.

### Section 11 recap

| Point | Detail |
|---|---|
| The import moved | `langchain_litellm`, not `langchain_community.chat_models`. Verified |
| Pass-through kwargs | go in `model_kwargs`, not as top level kwargs |
| Prefer `ChatLiteLLMRouter` | same code, and you inherit every S6 behaviour |
| Retry budgets multiply | LangChain retry x Router retry x fallback hops = 18 calls from one request |
| One retry budget per failure domain | provider to the Router, node code to `RetryPolicy`, whole graph to `recursion_limit` |
| Safety bypasses the verifier loop | do not add latency to the tightest latency budget |
| Wrap the checkpointer | a memory outage degrades the turn, it does not fail it |

---
# S12. Capstone: the whole Dispatch Copilot, failure tested

Every layer, wired, and then deliberately broken. This is the section to keep.

```mermaid
flowchart TB
    IN([request + channel + key]) --> PL["plugins S8<br/>redact, ledger, eligibility"]
    PL --> DEC["decision layer S7<br/>L1 hard, L3, L4, L2 constraint"]
    DEC --> RUNTIME{runtime}
    RUNTIME -- exploration --> STR["Strands agent S10<br/>tools, memory degradation"]
    RUNTIME -- regulated sequence --> LGD["LangGraph desk S11<br/>invariants, audit trail"]
    STR --> RT["Router S6<br/>LB, cooldowns, 3 fallback ladders"]
    LGD --> RT
    RT --> SDK["SDK S4/S5<br/>four R's, degrade never raise"]
    SDK --> BR[(Bedrock)]
    RT -.-> OBS["silent degradation<br/>cost ledger"]
```

In [ ]:
# ============================================================
# S12. TorchlightDesk: one object, every layer.
# ============================================================
class TorchlightDesk:
    def __init__(self, router, decision_layer, ledger):
        self.router, self.brain, self.ledger = router, decision_layer, ledger

    async def handle(self, text: str, channel: str = "consumer",
                     case_id: str = "TL-88421", canned=None, messages=None) -> dict:
        canned = canned or {}
        msgs = messages or [{"role": "user", "content": text}]
        report = {"route": None, "level": None, "answer": None,
                  "escalated": False, "degraded": [], "served_by": None}

        # 1. decision layer (S7). Failure here routes UP, never down.
        try:
            d = await self.brain.decide(msgs, channel=channel, canned=canned)
        except Exception as e:
            d = Decision("tl-deep", f"decision layer failed ({type(e).__name__}), safe-up", "fallback")
            report["degraded"].append("decision-layer")
        report["route"], report["level"] = d.group, d.level

        # 2. answer through the Router (S6), which carries retries and all three ladders
        try:
            # NOTE: a call-level mock_response would override the DEPLOYMENT-level
            # failure injection (the trap from S6.4) and silently defeat the failure
            # matrix below. So `answer` is only honoured when nothing is being broken.
            r = await self.router.acompletion(
                model=d.group, messages=msgs, max_tokens=300,
                metadata={"traffic_class": "safety" if d.group == "tl-guard" else "general"},
                **({"mock_response": canned["answer"]}
                   if (MOCK and "answer" in canned and not canned.get("_injected")) else {}))
            text_out = r.choices[0].message.content
            report["served_by"] = r._hidden_params.get("model_id")
        except Exception as e:
            report["degraded"].append(f"answer:{type(e).__name__}")
            report["answer"] = (f"I cannot reach the system right now. Your case is {case_id} "
                                f"and a human is joining.")
            return report

        # 3. verify and escalate once (S7.6). Safety skips the loop.
        if d.group != "tl-guard":
            ok, why = structural_verify(text_out)
            if not ok:
                nxt = promote(d.group)
                report["escalated"] = True
                report["degraded"].append(f"verify-fail:{why}")
                try:
                    r = await self.router.acompletion(
                        model=nxt, messages=msgs, max_tokens=300,
                        **({"mock_response": canned["escalated"]}
                           if (MOCK and "escalated" in canned and not canned.get("_injected")) else {}))
                    text_out = r.choices[0].message.content
                    report["route"], report["served_by"] = nxt, r._hidden_params.get("model_id")
                except Exception as e:
                    report["degraded"].append(f"escalation:{type(e).__name__}")

        report["answer"] = text_out
        return report

# A FRESH router. The global `router` has accumulated cooldowns from the failure demos
# in S6 to S8, which would make this section's deployment choices look random.
capstone_router = build_router()
desk = TorchlightDesk(capstone_router, TorchlightRouter(capstone_router), plugin)
print("TorchlightDesk assembled: plugins + decision layer + router + verify/escalate")

In [ ]:
rule("Capstone: five traffic classes, end to end")

CAPSTONE = [
    ("Where is my technician for TL-88421?", "consumer",
     {"answer": "Case TL-88421: EN-14 assigned, ETA 22 minutes, currently on the A34."}),
    ("Stranded on the M1 hard shoulder, two kids in the car, it is dark", "consumer",
     {"answer": "Case TL-88421: leave the vehicle, stand behind the barrier. Police notified, EN-14 en route."}),
    ("Grinding noise from the front left wheel, worse when braking", "consumer",
     {"l3": "triage", "answer": "Case TL-88421: likely worn pads or a seized caliper. EN-14 dispatched."}),
    ("Who pays for the tow under FL-2291?", "fleet",
     {"l3": "coverage", "answer": "Case TL-88421: FL-2291 clause 7.2 covers the tow, GBP 50 excess."}),
    ("uhh hello?", "consumer",
     {"l3": "nonsense", "l4": json.dumps({"group": "tl-fast", "confidence": 0.3, "reason": "no clear ask"}),
      "answer": "I need a bit more. What is happening with the vehicle?",
      "escalated": "Case TL-88421 is open. Tell me the fault and I will dispatch."}),
]

for text, ch, canned in CAPSTONE:
    rep = await desk.handle(text, channel=ch, canned=canned)
    flag = " ESC" if rep["escalated"] else "    "
    print(f"\n  {text[:56]}")
    print(f"    route {rep['route']:12s} [{rep['level']:>6s}]{flag}  served_by={rep['served_by']}")
    print(f"    {rep['answer'][:76]}")
    if rep["degraded"]:
        print(f"    degraded: {rep['degraded']}")

### 12.1 The failure injection matrix

The runbook you can **execute**. Each row breaks one thing and asserts the desk still serves the caller.

| # | Injected failure | Expected behaviour |
|---|---|---|
| 1 | `tl-fast` down in both regions | generic fallback ladder to `tl-deep`, answer arrives |
| 2 | Prompt exceeds `tl-fast` context | context ladder to `tl-balanced` (`lite-use1`), one hop |
| 3 | Classifier throttled | decision layer falls back, routes **up**, never to the cheap default |
| 4 | Every group down | degraded answer with a case reference, no exception to the caller |
| 5 | Session store unreachable | stateless turn, degradation flagged honestly |
| 6 | Safety keyword under a broken classifier | still `tl-guard`, because L1 is hard and runs first |

In [ ]:
rule("Failure injection matrix: break it on purpose")

async def scenario(n, label, router_kwargs, text, channel="consumer", canned=None, brain_router=None):
    r = build_router(**router_kwargs)
    d = TorchlightDesk(r, TorchlightRouter(brain_router or r), plugin)
    rep = await d.handle(text, channel=channel, canned=canned or {})
    served = rep["answer"] is not None and len(rep["answer"]) > 10
    print(f"\n  [{n}] {label}")
    print(f"      route={rep['route']}  served_by={rep['served_by']}  degraded={rep['degraded']}")
    print(f"      caller received an answer: {served}")
    print(f"      -> {rep['answer'][:70]}")
    return served

results = []
DOWN = "litellm.InternalServerError"

# No call-level `answer` in these rows. The deployment-level canned responses in
# torchlight_model_list() are what identify WHICH deployment actually served the call.
results.append(await scenario(1, "tl-fast down in both regions",
    {"fail": {"micro-use1": DOWN, "micro-usw2": DOWN}, "num_retries": 0},
    "Where is my technician for TL-88421?"))

results.append(await scenario(2, "prompt overflows tl-fast context",
    {"fail": {"micro-use1": "litellm.ContextWindowExceededError",
              "micro-usw2": "litellm.ContextWindowExceededError"}, "num_retries": 0},
    "Where is my technician for TL-88421?"))

results.append(await scenario(3, "classifier throttled, no safety keyword",
    {"num_retries": 0},
    "the thing is making the noise again", canned={"l3": None, "l4": None}))

results.append(await scenario(4, "every group down",
    {"fail": {k: DOWN for k in ["micro-use1", "micro-usw2", "lite-use1", "haiku-use1", "sonnet-use1"]},
     "num_retries": 0},
    "Where is my technician for TL-88421?"))

results.append(await scenario(6, "safety keyword while the classifier is broken",
    {"num_retries": 0},
    "Fire in the engine bay on the hard shoulder, kids in the car",
    canned={"l3": None, "l4": None}))

print(f"\n  callers served in {sum(results)}/{len(results)} injected failures.")
print("  Scenario 4 is the important one: everything was down and the caller still got")
print("  a case reference and a human handoff instead of a stack trace.")

In [ ]:
rule("SELF-CHECK: run this before teaching or shipping")

checks = []
def chk(name, cond, detail=""):
    checks.append((name, bool(cond), detail))

# S4
chk("model info resolves for every fleet model",
    all(litellm.get_model_info(m).get("max_input_tokens") for _, m in FLEET))
chk("context window uses max_input_tokens not get_max_tokens",
    litellm.get_model_info(HAIKU)["max_input_tokens"] != litellm.get_max_tokens(HAIKU),
    f"{litellm.get_model_info(HAIKU)['max_input_tokens']} vs {litellm.get_max_tokens(HAIKU)}")
chk("nova-micro has no vision (capability routing is real)", litellm.supports_vision(MICRO) is False)

# S5
chk("ContextWindowExceededError is a BadRequestError (catch order matters)",
    issubclass(litellm.ContextWindowExceededError, litellm.BadRequestError))
chk("litellm.Timeout is NOT a litellm.APIConnectionError (list it explicitly)",
    not issubclass(litellm.Timeout, litellm.APIConnectionError))
chk("litellm.APIError is NOT a usable catch-all",
    not isinstance(litellm.RateLimitError(message="m", model=HAIKU, llm_provider="bedrock"),
                   litellm.APIError))

# S6
_r = build_router(fail={"micro-use1": DOWN, "micro-usw2": DOWN}, num_retries=0)
_res = await _r.acompletion(model="tl-fast", messages=[{"role": "user", "content": "x"}], max_tokens=20)
chk("generic fallback ladder reaches a healthy group", _res._hidden_params["model_id"] == "haiku-use1",
    _res._hidden_params["model_id"])

_r2 = build_router(fail={"micro-use1": "litellm.ContextWindowExceededError",
                         "micro-usw2": "litellm.ContextWindowExceededError"}, num_retries=0)
_res2 = await _r2.acompletion(model="tl-fast", messages=[{"role": "user", "content": "x"}], max_tokens=20)
chk("context ladder jumps to the 300k group in one hop",
    _res2._hidden_params["model_id"] == "lite-use1", _res2._hidden_params["model_id"])

# S7
_d = await TorchlightRouter(router).decide(
    [{"role": "user", "content": "kids in the car on the hard shoulder"}], canned={"l3": "status"})
chk("L1 hard safety rule beats a cheap classifier label", _d.group == "tl-guard", _d.group)
_d2 = await TorchlightRouter(router).decide(PHOTO, canned={"l3": "status"})
chk("L2 capability constraint promotes off a non-vision group",
    GROUP_CAPS[_d2.group]["vision"] is True, _d2.group)

# S8
chk("litellm.max_budget does NOT enforce (build your own guard)", litellm.max_budget in (0.0, None))
chk("silent degradation was detected by the fallback hook", len(plugin.silent_degradations) > 0)

# S11
try:
    from langchain_community.chat_models import ChatLiteLLM as _old
    chk("langchain_community.ChatLiteLLM is gone", False, "it imported, check your versions")
except Exception:
    chk("langchain_community.ChatLiteLLM is gone (use langchain_litellm)", True)

width = max(len(n) for n, _, _ in checks)
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:<{width}}  {detail}")
print(f"\n  {sum(1 for _, ok, _ in checks if ok)}/{len(checks)} checks passed")

---
## 12.2 Your turn

Five TODO cells. Each one is a real design decision you will face. The answer key follows, so commit to an answer before you scroll.

**Language:** Python
**Topics:** LiteLLM Router, fallback ladders, routing levels, plugins, cost control
**Level:** applied

In [ ]:
# ============================================================
# EXERCISE 1. Torchlight adds a Spanish speaking desk. Nova Micro handles Spanish
# poorly; Nova Lite handles it well. Write the L1 rule.
#
# Constraint: this is a CAPABILITY decision, so it must behave like L2, not L3.
# It must be a HARD decision that a classifier cannot override.
# ============================================================
SPANISH_MARKERS = ["averiado", "grua", "carretera", "no arranca", "bateria"]

def l1_language(text: str) -> Optional[Decision]:
    # TODO: return a hard Decision routing to "tl-balanced" when Spanish markers appear.
    # Return None otherwise. Include the matched markers in the reason.
    pass

for t in ["mi coche no arranca en la carretera", "my car will not start"]:
    print(f"  {t[:38]:40s} -> {l1_language(t)}")

In [ ]:
# ============================================================
# EXERCISE 2. A new group `tl-vision` is added for photo diagnosis.
# Write its three fallback ladder entries.
#
# Facts you need: tl-vision is 128k and vision capable. tl-balanced is 300k and
# vision capable. tl-deep is 200k and vision capable. tl-fast has NO vision.
# ============================================================
vision_fallbacks = {
    "fallbacks":                [],    # TODO
    "context_window_fallbacks": [],    # TODO
    "content_policy_fallbacks": [],    # TODO
}
print(vision_fallbacks)

In [ ]:
# ============================================================
# EXERCISE 3. Write a plugin that alerts when the fallback rate crosses 10%
# over a rolling window of 50 requests. This is the silent degradation alarm.
# ============================================================
class FallbackRateAlarm(CustomLogger):
    def __init__(self, window=50, threshold=0.10):
        super().__init__()
        self.window, self.threshold = window, threshold
        self.recent: list[int] = []      # 1 = rescued by a fallback, 0 = normal
        self.alarms: list[str] = []

    async def async_log_success_event(self, kwargs, response_obj, start_time, end_time):
        # TODO: record a 0, trim to the window
        pass

    def log_success_fallback_event(self, original_model_group, kwargs, original_exception):
        # TODO: record a 1, trim to the window, and append to self.alarms when the
        #       rate crosses the threshold AND the window is full
        pass

print("FallbackRateAlarm defined")

In [ ]:
# ============================================================
# EXERCISE 4. Torchlight's finance team wants a hard answer:
# "at what escalation rate does our tl-balanced -> tl-deep cascade stop paying?"
# Compute it. Then say what you would monitor instead.
# ============================================================
r_star = None   # TODO: compute using tier_cost() and the formula from S7.6
print("break-even escalation rate:", r_star)

In [ ]:
# ============================================================
# EXERCISE 5. Spot the bug. This handler shipped and quietly failed for six weeks.
# Name the bug, then fix it.
# ============================================================
def buggy_handler(model, messages):
    try:
        return litellm.completion(model=model, messages=messages)
    except litellm.APIError:
        return "temporary problem, retrying"
    except litellm.BadRequestError:
        return "bad request"
    except litellm.ContextWindowExceededError:
        return "too long, using a bigger model"

# TODO: write the corrected version below as fixed_handler(model, messages)

### Answer key

**Exercise 1.** Language is a capability constraint, so it must be `hard=True` and it must run in the L1 pass where a classifier cannot overrule it. Routing to `tl-balanced` rather than `tl-deep` is deliberate: Nova Lite is both multilingual and cheaper than Haiku, so this is a capability promotion, not a cost escalation.

**Exercise 2.** The three ladders differ because the three failures differ.

| Ladder | Entry | Why |
|---|---|---|
| `fallbacks` | `{"tl-vision": ["tl-deep"]}` | a healthier vision capable group |
| `context_window_fallbacks` | `{"tl-vision": ["tl-balanced"]}` | 128k to 300k, and still vision capable |
| `content_policy_fallbacks` | `{"tl-vision": ["tl-guard"]}` | different alignment, still vision capable |

The trap in this exercise: `tl-fast` is a valid destination on paper (it is healthy and cheap) and a **wrong** destination in practice, because it cannot see images. Capability constraints propagate through the ladders. A fallback destination that cannot do the job is not a fallback.

**Exercise 3.** Both hooks write to the same window. The alarm only fires when the window is full, otherwise the first fallback in a fresh process reads as a 100% failure rate and pages someone at 03:00 over a single blip.

**Exercise 4.** `r* = 1 - (tl-balanced cost / tl-deep cost)`, which the cell below computes as roughly 95%. **The honest answer to finance is that cost is not the constraint here, latency is.** At a 95% break even the cascade essentially always pays, so the number to monitor is p95 latency on the escalated path, plus the escalation rate as a **quality** signal: a rising escalation rate means the cheap tier is getting worse at the task, which is a model or prompt problem wearing a cost costume.

**Exercise 5.** Three bugs, in order of severity.

1. `except litellm.APIError` catches **nothing** (verified in S5.2), so throttling, timeouts and 500s all escape this handler entirely and became unhandled 500s for six weeks.
2. `except litellm.BadRequestError` sits **above** `except litellm.ContextWindowExceededError`, so the context branch is dead code even after you fix bug 1.
3. Even corrected, this handler retries nothing and re-routes nothing. The correct fix is not a better `try` block, it is a `Router` with `fallbacks` and `context_window_fallbacks`.

In [ ]:
rule("Answer key: runnable")

# --- 1 ---
def l1_language_solved(text: str) -> Optional[Decision]:
    hits = [m for m in SPANISH_MARKERS if re.search(rf"\b{re.escape(m)}\b", text, re.I)]
    if hits:
        return Decision("tl-balanced", f"spanish markers {hits}", "L1", 1.0, hard=True)
    return None

for t in ["mi coche no arranca en la carretera", "my car will not start"]:
    d = l1_language_solved(t)
    print(f"  1) {t[:34]:36s} -> {d.group if d else 'no match':12s} hard={d.hard if d else '-'}")

# --- 2 ---
vision_fallbacks_solved = {
    "fallbacks":                [{"tl-vision": ["tl-deep"]}],
    "context_window_fallbacks": [{"tl-vision": ["tl-balanced"]}],
    "content_policy_fallbacks": [{"tl-vision": ["tl-guard"]}],
}
print("\n  2)", json.dumps(vision_fallbacks_solved))
print("     note tl-fast appears in NONE of them: no vision, so it cannot do the job")

# --- 3 ---
class FallbackRateAlarmSolved(CustomLogger):
    def __init__(self, window=50, threshold=0.10):
        super().__init__()
        self.window, self.threshold = window, threshold
        self.recent, self.alarms = [], []

    def _record(self, value: int):
        self.recent.append(value)
        self.recent = self.recent[-self.window:]
        if len(self.recent) == self.window:
            rate = sum(self.recent) / self.window
            if rate > self.threshold:
                self.alarms.append(f"fallback rate {rate:.0%} over last {self.window}")

    async def async_log_success_event(self, kwargs, response_obj, start_time, end_time):
        self._record(0)

    def log_success_fallback_event(self, original_model_group, kwargs, original_exception):
        self._record(1)

alarm = FallbackRateAlarmSolved(window=10, threshold=0.10)
for i in range(10):
    alarm._record(1 if i < 3 else 0)
print("\n  3) alarms:", alarm.alarms)
print("     window full before firing, so a single early blip never pages anyone")

# --- 4 ---
r_star_solved = 1 - tier_cost(LITE) / tier_cost(HAIKU)
print(f"\n  4) tl-balanced -> tl-deep break-even r* = {r_star_solved:.0%}")
print("     so cost is not the binding constraint. Monitor p95 on the escalated path")
print("     and treat a RISING escalation rate as a quality regression, not a cost event.")

# --- 5 ---
def fixed_handler(model, messages, **kw):
    """Correct ordering, no litellm.APIError, and an honest terminal state."""
    try:
        return litellm.completion(model=model, messages=messages, **kw)
    except litellm.ContextWindowExceededError:
        return "RESHAPE: route to a larger context group"
    except litellm.ContentPolicyViolationError:
        return "RESHAPE: route to a differently aligned group"
    except litellm.BadRequestError:
        return "REFUSE: this is a bug in our request"
    except (litellm.RateLimitError, litellm.Timeout, litellm.APIConnectionError):
        return "RETRY then RE-ROUTE"
    except litellm.AuthenticationError:
        return "REFUSE: page a human, config is wrong"

print("\n  5) corrected handler on four injected failures:")
for inj, label in [("litellm.ContextWindowExceededError", "context overflow"),
                   ("litellm.RateLimitError", "throttled"),
                   (litellm.AuthenticationError(message="m", model=MICRO, llm_provider="bedrock"), "auth"),
                   (litellm.ContentPolicyViolationError(message="m", model=MICRO, llm_provider="bedrock"), "refusal")]:
    print(f"     {label:18s} -> {fixed_handler(MICRO, [{'role':'user','content':'x'}], mock_response=inj)}")
print("\n     And the real fix is a Router, not a better try block.")

---
# S13. Where this fails

The honest section. Nothing above is free, and a few of these will bite you.

| Design | Fails when | What you actually do |
|---|---|---|
| **Model groups as job names** | a new use case does not fit any group and someone invents `tl-fast-but-longer` | cap the group count. Four to six. More groups means the abstraction has leaked back into the application |
| **Cascade escalation** | the escalation rate drifts up, or the tier price ratio narrows | alert on the escalation rate. Recompute `r* = 1 - cheap/expensive` whenever you change a tier |
| **LLM router (L4)** | the routing model is upgraded and its judgement shifts | pin the routing model version. A router that changes behaviour on a provider deploy is not a control plane |
| **Latency-based routing** | a cold deployment has no samples and gets starved, or all traffic herds onto one node | pair with `simple-shuffle` weights during warm up, and never use it as your only strategy |
| **`enable_pre_call_checks`** | it estimates tokens; a request near the boundary can still overflow at the provider | keep `context_window_fallbacks` configured as well. Belt and braces is correct here, unlike retries |
| **Prompt caching** | a live fact gets cached, or two tenants share a key | never cache class A. Namespace by tenant. Cache reasoning, not facts |
| **Proxy as gateway** | it becomes a single point of failure, or a bottleneck at very high QPS | two or more replicas, shared Redis, and a documented break glass path that talks to Bedrock directly |
| **Silent degradation detection** | nobody reads the metric | alert on `log_success_fallback_event` rate, not just error rate. A green dashboard during a fallback storm is the failure |
| **Offline `mock_response` testing** | it does not exercise provider quirks: token limits, tool schema differences, real latency | mocks test **your control flow**. A weekly smoke test against real Bedrock tests **their behaviour**. You need both |
| **The whole architecture** | you have one model, one team and one app | you do not need any of this. Use `litellm.completion` and revisit when a second consumer appears |

**The one that catches everyone.** All of this optimises **routing**. None of it fixes a bad prompt, a missing tool or an unclear task definition. If the cheap model is wrong 40% of the time on a task the strong model gets right 45% of the time, the answer is not a cascade. It is that the task is not specified well enough for either.

### 13.1 What changes in production

Everything in this notebook runs with `MOCK = True` and hardcoded values. The delta to production:

| Notebook | Production |
|---|---|
| `MOCK = True`, canned responses | real Bedrock, and a CI suite that keeps the mocked control flow tests |
| AWS creds from env or `aws configure` | IAM roles on the task or pod, never long lived keys |
| Region and model ids in Python literals | `config.yaml`, loaded by the proxy, versioned in git, reviewed |
| `Cache(type="local")` | `Cache(type="redis")`, shared and namespaced by tenant |
| in process cost ledger | proxy spend logs in Postgres, per virtual key |
| `print()` traces | structured logs with a request id, correlated across Strands hooks and LiteLLM callbacks |
| `InMemorySaver` | `PostgresSaver`, wrapped by the resilient pattern in S11.2 |
| one process | two or more proxy replicas, shared Redis for rpm and tpm accounting |
| plugins in the notebook | a module loaded by dotted path in `litellm_settings.callbacks` |

**And the one operational habit worth adopting today:** run the `model_list` validation from S9.1 in CI on every config change. A typo in a Bedrock model string costs 200ms to catch there and a 3am page to catch in production.

### 13.2 Decision cards

Cut these out. They are the notebook in one page.

**Card 1: which LiteLLM hat**

| Situation | Use |
|---|---|
| a script, one model | `litellm.completion` |
| one service, one team, Python, needs failover | `litellm.Router` |
| second team, second language, or a finance question | Proxy |
| the proxy is the mandatory path and bypassing it is a violation | it is now a Gateway |

**Card 2: which routing level**

| Signal | Level |
|---|---|
| safety, compliance, or anything a non engineer must audit | L1 heuristic, hard, runs first |
| token count, image present, tools needed | L2 constraint, runs last, can only promote |
| phrasing varies more than a keyword list holds | L3 small model classifier |
| the request needs splitting or genuine judgement | L4 LLM router, pinned version |
| you can verify the cheap answer cheaply | L5 cascade, and monitor the escalation rate |

**Card 3: which fallback ladder**

| Exception | Ladder | Destination |
|---|---|---|
| `InternalServerError`, `ServiceUnavailableError`, `RateLimitError` | `fallbacks` | a healthier group |
| `ContextWindowExceededError` | `context_window_fallbacks` | a bigger context group |
| `ContentPolicyViolationError` | `content_policy_fallbacks` | a differently aligned model |
| `AuthenticationError`, `BadRequestError`, `NotFoundError` | none | refuse, page a human, it is a bug |

**Card 4: which load balancing strategy**

| Symptom | Strategy |
|---|---|
| nothing wrong, you want a canary | `simple-shuffle` with `weight` |
| long variable duration calls queueing | `least-busy` |
| one region is slow but not failing | `latency-based-routing` plus a tight `timeout` |
| same quality, different price | `cost-based-routing` |
| hard provider quotas at high volume | `usage-based-routing-v2` plus shared Redis |

**Card 5: where does this retry belong**

| Failure | Owner | Never also in |
|---|---|---|
| provider throttled or 5xx | LiteLLM Router `num_retries` and `fallbacks` | LangChain `with_retry`, agent code |
| your node's own code, a tool, a parse | LangGraph `RetryPolicy` | the Router |
| the whole graph is wedged | `recursion_limit` plus a timeout above it | anywhere else |

**Card 6: the four questions before adding a model group**

1. Does an existing group already satisfy the hard constraints (context, vision, tools)?
2. Is this a **traffic** decision or a **behaviour** decision? Traffic goes in the Router, behaviour goes in the application.
3. What is its fallback destination, in all three ladders?
4. Who is allowed to call it, and what happens to the budget when they do?

---
### Closing

Run the notebook top to bottom with `MOCK = True` and nothing is spent. Flip to `MOCK = False` with Bedrock access and the same code runs against real models.

**The single sentence:** LiteLLM is not an abstraction over models, it is the place where the decisions about **which model, what happens when it fails, and who pays** stop being scattered through your application and start being one configurable layer that your agent frameworks sit on top of, and can be swapped out from underneath.